# Liquidador de retenciones Colombia 2026

**Implementación en Python del diseño funcional** de `docs/diseno-funcional.md`.

Este notebook convierte el diseño en un motor ejecutable:

1. Modelo de datos (comprador, proveedor, operación, regla, liquidación).
2. Tablas normativas con vigencia y fundamento legal.
3. Motor de decisión: retefuente, reteIVA y reteICA por separado.
4. Asientos de causación, pago y pago de retenciones.
5. Certificados, checklist documental y registro de auditoría.
6. Los 12 casos de prueba de la sección 10 y los criterios de aceptación.

> **Advertencia:** es un proyecto educativo. No reemplaza al contador ni la
> tabla oficial de retenciones. La UVT, las tarifas y la vigencia deben
> verificarse antes de usar el motor en una empresa real.

**Requisitos:** Python 3.10+ y Jupyter. `pandas` es opcional: solo se usa
para mostrar las tablas; el motor funciona con la biblioteca estándar.

---

## 1. Configuración y utilidades

In [1]:
from __future__ import annotations

import hashlib
import json
import unicodedata
from dataclasses import asdict, dataclass, field
from datetime import date, datetime
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum
from typing import Any, Iterable, Sequence

try:
    import pandas as pd
    pd.set_option("display.max_columns", 30)
    pd.set_option("display.width", 180)
    pd.set_option("display.max_colwidth", 45)
except ImportError:  # el motor funciona sin pandas; solo se pierde la tabla bonita
    pd = None

try:
    from IPython.display import display as _display
except ImportError:
    _display = print


def mostrar(objeto: Any) -> None:
    """Muestra un DataFrame en Jupyter y lo imprime si se ejecuta como script."""
    if pd is not None and isinstance(objeto, pd.DataFrame) and _display is print:
        print(objeto.to_string(index=False))
    else:
        _display(objeto)


def D(valor: Any) -> Decimal:
    """Convierte a Decimal sin pasar por float (evita 0.1 + 0.2)."""
    if isinstance(valor, Decimal):
        return valor
    if isinstance(valor, bool):
        raise TypeError("Un booleano no es un valor monetario.")
    return Decimal(str(valor))


CENTAVO = D("0.01")
CERO = D("0")


def redondear(valor: Any) -> Decimal:
    """Política única de redondeo: dos decimales, half-up."""
    return D(valor).quantize(CENTAVO, rounding=ROUND_HALF_UP)


def dinero(valor: Any) -> Decimal:
    return redondear(valor)


def cop(valor: Any) -> str:
    """Formato de moneda colombiana para leer los resultados."""
    numero = f"{D(valor):,.2f}"
    return "$" + numero.replace(",", "X").replace(".", ",").replace("X", ".")


def porcentaje(valor: Any) -> str:
    numero = f"{D(valor) * 100:.4f}".rstrip("0").rstrip(".")
    return numero.replace(".", ",") + "%"


def normalizar(texto: str) -> str:
    """Minúsculas y sin tildes, para comparar municipios y actividades."""
    sin_tildes = "".join(
        c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c)
    )
    return sin_tildes.strip().lower()


UVT_2026 = D("52374")
VERSION_TABLAS = "2026.1"
TABLA_UVT: dict[int, Decimal] = {2026: UVT_2026}

IVA_GENERAL = D("0.19")
IVA_DIFERENCIAL = D("0.05")
IVA_EXCLUIDO = D("0")
TARIFAS_IVA_PERMITIDAS = {IVA_GENERAL, IVA_DIFERENCIAL, IVA_EXCLUIDO}

TARIFA_RETEIVA = D("0.15")
MIN_RETEIVA_SERVICIOS_UVT = D("2")
MIN_RETEIVA_COMPRAS_UVT = D("10")

## 2. Enumeraciones y modelo de datos

In [2]:
class TipoPersona(str, Enum):
    JURIDICA = "Jurídica"
    NATURAL = "Natural"


class Regimen(str, Enum):
    ORDINARIO = "Ordinario"
    SIMPLE = "SIMPLE"
    ESPECIAL = "Especial"


class TipoDocumento(str, Enum):
    FACTURA = "Factura electrónica"
    NOTA_CREDITO = "Nota crédito"
    DOCUMENTO_SOPORTE = "Documento soporte"


class Concepto(str, Enum):
    SERVICIO_GENERAL = "Servicio general"
    HONORARIOS = "Honorarios"
    ARRENDAMIENTO_VIVIENDA = "Arrendamiento de vivienda"
    ARRENDAMIENTO_MUEBLES = "Arrendamiento de bienes muebles"
    COMPRA_BIENES = "Compra de bienes"
    TRANSPORTE_CARGA = "Transporte de carga"
    HOTELES_RESTAURANTES = "Hoteles y restaurantes"


class Severidad(str, Enum):
    INFO = "Info"
    ALERTA = "Alerta"
    BLOQUEO = "Bloqueo"


@dataclass(frozen=True)
class Alerta:
    codigo: str
    mensaje: str
    severidad: Severidad = Severidad.ALERTA
    campo: str | None = None


class CalculoBloqueado(Exception):
    """El motor no calcula cuando falta un dato, una regla o el caso está fuera de alcance."""

    def __init__(self, mensaje: str, alertas: Sequence[Alerta] = ()) -> None:
        super().__init__(mensaje)
        self.mensaje = mensaje
        self.alertas = list(alertas)


class RevisionHumana(CalculoBloqueado):
    """Caso fuera del alcance de la v1: se deriva a revisión humana."""


class ReglaNoEncontrada(CalculoBloqueado):
    """No hay regla vigente para el concepto, la fecha o el municipio."""


@dataclass(frozen=True)
class Comprador:
    nit: str
    razon_social: str
    tipo: TipoPersona
    agente_retefuente: bool
    agente_reteiva: bool
    agente_reteica: bool
    municipio: str
    responsable_iva: bool
    obligado_contabilidad: bool
    regimen: Regimen = Regimen.ORDINARIO
    direccion: str = ""
    gran_contribuyente: bool = False
    entidad_estatal: bool = False
    emisor_tarjetas: bool = False


@dataclass(frozen=True)
class Proveedor:
    nit: str
    razon_social: str
    tipo: TipoPersona
    residente: bool
    declarante: bool
    responsable_iva: bool
    simple: bool
    autorretenedor: bool
    autorretencion_especial: bool
    gran_contribuyente: bool
    actividad_ciiu: str
    municipio: str
    rut_recibido: bool
    soporte_no_declarante: bool
    actividad_gravada_ica: bool = True


@dataclass(frozen=True)
class Operacion:
    tipo_documento: TipoDocumento
    numero: str
    fecha_emision: date
    fecha_pago: date
    concepto: Concepto
    descripcion: str
    base: Decimal | None
    tarifa_iva: Decimal
    iva: Decimal | None
    total: Decimal | None
    total_incluye_iva: bool
    descuentos: Decimal
    moneda: str
    trm: Decimal | None
    forma_pago: str
    valor_pagado: Decimal | None
    retencion_ya_practicada: bool
    factura_validada: bool
    orden_compra: bool
    acta_recibido: bool
    periodo_cerrado: bool
    iva_excluido: bool = False
    centro_costo: str = "ADMIN"


@dataclass(frozen=True)
class DatosFactura:
    base_bruta: Decimal
    descuentos: Decimal
    base: Decimal
    iva: Decimal
    total: Decimal
    tarifa_iva: Decimal
    moneda_original: str
    trm: Decimal
    factor_conversion: Decimal


@dataclass(frozen=True)
class ReglaRetefuente:
    id: str
    concepto: Concepto
    condicion: str
    tarifa: Decimal
    minimo_uvt: Decimal
    base: str
    fundamento: str
    vigencia_desde: date
    vigencia_hasta: date
    version: str = VERSION_TABLAS

    def vigente(self, fecha: date) -> bool:
        return self.vigencia_desde <= fecha <= self.vigencia_hasta


@dataclass(frozen=True)
class ReglaICA:
    municipio: str
    actividad: str
    tarifa: Decimal
    minimo_uvt: Decimal
    fundamento: str
    vigencia_desde: date
    vigencia_hasta: date
    version: str = VERSION_TABLAS

    def vigente(self, fecha: date) -> bool:
        return self.vigencia_desde <= fecha <= self.vigencia_hasta


@dataclass(frozen=True)
class FilaDecision:
    numero: int
    comprador: str
    proveedor: str
    concepto: Concepto
    aplica_rf: bool
    tarifa_rf: Decimal | None
    aplica_reteiva: str
    aplica_reteica: str


@dataclass
class LineaAsiento:
    cuenta: str
    nombre: str
    debito: Decimal
    credito: Decimal
    tercero: str
    centro_costo: str = ""
    descripcion: str = ""
    documento: str = ""


@dataclass
class Asiento:
    tipo: str
    fecha: date
    documento: str
    descripcion: str
    lineas: list[LineaAsiento] = field(default_factory=list)

    @property
    def total_debito(self) -> Decimal:
        return sum((linea.debito for linea in self.lineas), CERO)

    @property
    def total_credito(self) -> Decimal:
        return sum((linea.credito for linea in self.lineas), CERO)

    @property
    def diferencia(self) -> Decimal:
        return redondear(self.total_debito - self.total_credito)

    @property
    def cuadra(self) -> bool:
        return abs(self.diferencia) < D("0.005")


@dataclass
class CertificadoRetencion:
    tipo: str
    numero: str
    anio_gravable: int
    agente_nit: str
    agente_nombre: str
    retenido_nit: str
    retenido_nombre: str
    concepto: str
    base: Decimal
    tarifa: Decimal
    valor: Decimal
    fundamento: str
    ciudad: str
    fecha_expedicion: date
    base_sin_iva: Decimal | None = None
    iva: Decimal | None = None

    def validar(self) -> list[str]:
        faltantes: list[str] = []
        obligatorios = {
            "tipo": self.tipo,
            "número": self.numero,
            "NIT del agente": self.agente_nit,
            "nombre del agente": self.agente_nombre,
            "NIT del retenido": self.retenido_nit,
            "nombre del retenido": self.retenido_nombre,
            "concepto": self.concepto,
            "fundamento": self.fundamento,
            "ciudad": self.ciudad,
        }
        for nombre, valor in obligatorios.items():
            if valor is None or str(valor).strip() == "":
                faltantes.append(nombre)
        if self.base < 0 or self.valor < 0 or self.tarifa < 0:
            faltantes.append("valores negativos")
        if self.tipo == "ReteIVA" and (self.base_sin_iva is None or self.iva is None):
            faltantes.append("base sin IVA e IVA discriminados")
        return faltantes

    def requisitos_art_381(self) -> dict[str, Any]:
        """Los siete datos que debe explicar un certificado de renta."""
        return {
            "1. Nombre y NIT del agente retenedor": f"{self.agente_nombre} / {self.agente_nit}",
            "2. Nombre y NIT del retenido": f"{self.retenido_nombre} / {self.retenido_nit}",
            "3. Concepto": self.concepto,
            "4. Base": self.base,
            "5. Tarifa": self.tarifa,
            "6. Valor retenido": self.valor,
            "7. Año gravable": self.anio_gravable,
        }


@dataclass
class ItemChecklist:
    documento: str
    obligatorio: bool
    recibido: bool
    observacion: str = ""


@dataclass
class RegistroAuditoria:
    usuario: str
    fecha_hora: datetime
    version_tabla: str
    uvt: Decimal
    uvt_anio: int
    regla_id: str
    fundamento: str
    hash_entrada: str
    entrada_json: str
    resultado_json: str
    documentos: tuple[str, ...] = ()
    cambios: tuple[str, ...] = ()


@dataclass
class Liquidacion:
    comprador: Comprador
    proveedor: Proveedor
    operacion: Operacion
    datos: DatosFactura
    uvt: Decimal
    uvt_anio: int
    factor_pago: Decimal
    base_pago: Decimal
    iva_pago: Decimal
    total_pago: Decimal
    regla_rf: ReglaRetefuente | None
    tarifa_rf: Decimal
    base_rf: Decimal
    retefuente: Decimal
    minimo_rf_uvt: Decimal
    minimo_rf_valor: Decimal
    cumple_minimo_rf: bool
    aplica_reteiva: bool
    tarifa_reteiva: Decimal
    base_reteiva: Decimal
    reteiva: Decimal
    minimo_reteiva_uvt: Decimal
    minimo_reteiva_valor: Decimal
    cumple_minimo_reteiva: bool
    regla_ica: ReglaICA | None
    aplica_reteica: bool
    tarifa_reteica: Decimal
    base_reteica: Decimal
    reteica: Decimal
    minimo_reteica_uvt: Decimal
    minimo_reteica_valor: Decimal
    cumple_minimo_reteica: bool
    total_retenciones: Decimal
    neto: Decimal
    fila_decision: FilaDecision | None
    alertas: list[Alerta] = field(default_factory=list)
    asientos: list[Asiento] = field(default_factory=list)
    certificados: list[CertificadoRetencion] = field(default_factory=list)
    checklist: list[ItemChecklist] = field(default_factory=list)
    auditoria: RegistroAuditoria | None = None

    @property
    def asientos_cuadran(self) -> bool:
        return all(asiento.cuadra for asiento in self.asientos)

    @property
    def puede_pagar(self) -> bool:
        if any(a.severidad == Severidad.BLOQUEO for a in self.alertas):
            return False
        return all(item.recibido for item in self.checklist if item.obligatorio)

    def resumen(self) -> dict[str, Any]:
        return {
            "base_factura": self.datos.base,
            "iva_factura": self.datos.iva,
            "total_factura": self.datos.total,
            "factor_pago": self.factor_pago,
            "base_pago": self.base_pago,
            "iva_pago": self.iva_pago,
            "total_pago": self.total_pago,
            "concepto_rf": self.regla_rf.concepto.value if self.regla_rf else None,
            "regla_rf": self.regla_rf.id if self.regla_rf else None,
            "tarifa_rf": self.tarifa_rf,
            "base_rf": self.base_rf,
            "retefuente": self.retefuente,
            "aplica_reteiva": self.aplica_reteiva,
            "base_reteiva": self.base_reteiva,
            "reteiva": self.reteiva,
            "aplica_reteica": self.aplica_reteica,
            "base_reteica": self.base_reteica,
            "reteica": self.reteica,
            "total_retenciones": self.total_retenciones,
            "neto": self.neto,
            "fila_decision": self.fila_decision.numero if self.fila_decision else None,
            "uvt": self.uvt,
            "uvt_anio": self.uvt_anio,
            "version_tabla": VERSION_TABLAS,
            "asientos_cuadran": self.asientos_cuadran,
        }

## 3. Tablas normativas

La tabla de retefuente se parte por concepto y condición del proveedor. Las
filas de servicios incluyen la línea de tiempo del Decreto 572 de 2025 y de
la suspensión de 2026: el mínimo cambia según la fecha. Las fechas son
inclusivas.

In [3]:
def _regla_rf(
    id_: str,
    concepto: Concepto,
    condicion: str,
    tarifa: str,
    minimo_uvt: str,
    desde: date,
    hasta: date,
    fundamento: str,
) -> ReglaRetefuente:
    return ReglaRetefuente(
        id=id_,
        concepto=concepto,
        condicion=condicion,
        tarifa=D(tarifa),
        minimo_uvt=D(minimo_uvt),
        base="Valor antes de IVA, menos descuentos y devoluciones",
        fundamento=fundamento,
        vigencia_desde=desde,
        vigencia_hasta=hasta,
    )


_INICIO_2026 = date(2026, 1, 1)
_FIN_2026 = date(2026, 12, 31)
_CORTE_SUSPENSION = date(2026, 5, 7)
_INICIO_SUSPENSION = date(2026, 5, 8)
_FIN_SUSPENSION = date(2026, 6, 30)
_INICIO_D572 = date(2026, 7, 1)

TABLA_RETEFUENTE: tuple[ReglaRetefuente, ...] = (
    # Servicios generales: 4% declarante / 6% no declarante.
    # El mínimo cambia: 2 UVT, luego 4 UVT por la suspensión, y 2 UVT desde 1/07/2026.
    _regla_rf("RF-SERV-001", Concepto.SERVICIO_GENERAL, "declarante", "0.04", "2", _INICIO_2026, _CORTE_SUSPENSION, "Art. 392 ET; Decreto 572/2025; UVT 2026."),
    _regla_rf("RF-SERV-002", Concepto.SERVICIO_GENERAL, "no declarante", "0.06", "2", _INICIO_2026, _CORTE_SUSPENSION, "Art. 392 ET; Decreto 572/2025; UVT 2026."),
    _regla_rf("RF-SERV-003", Concepto.SERVICIO_GENERAL, "declarante", "0.04", "4", _INICIO_SUSPENSION, _FIN_SUSPENSION, "Suspensión temporal de los arts. 2–8 del Decreto 572/2025; UVT 2026."),
    _regla_rf("RF-SERV-004", Concepto.SERVICIO_GENERAL, "no declarante", "0.06", "4", _INICIO_SUSPENSION, _FIN_SUSPENSION, "Suspensión temporal de los arts. 2–8 del Decreto 572/2025; UVT 2026."),
    _regla_rf("RF-SERV-005", Concepto.SERVICIO_GENERAL, "declarante", "0.04", "2", _INICIO_D572, _FIN_2026, "Art. 392 ET; Decreto 572/2025; efectos desde 1/07/2026; UVT 2026."),
    _regla_rf("RF-SERV-006", Concepto.SERVICIO_GENERAL, "no declarante", "0.06", "2", _INICIO_D572, _FIN_2026, "Art. 392 ET; Decreto 572/2025; efectos desde 1/07/2026; UVT 2026."),
    # Honorarios: sin mínimo de retefuente.
    _regla_rf("RF-HON-001", Concepto.HONORARIOS, "declarante", "0.11", "0", _INICIO_2026, _FIN_2026, "Art. 392 ET; Art. 383 ET (excepción de 3.300 UVT)."),
    _regla_rf("RF-HON-002", Concepto.HONORARIOS, "no declarante", "0.10", "0", _INICIO_2026, _FIN_2026, "Art. 392 ET; Art. 383 ET (excepción de 3.300 UVT)."),
    # Arrendamiento de vivienda: 3,5%, mínimo 10 UVT; IVA excluido.
    _regla_rf("RF-ARR-001", Concepto.ARRENDAMIENTO_VIVIENDA, "declarante", "0.035", "10", _INICIO_2026, _FIN_2026, "Art. 392 ET; Art. 387 ET; UVT 2026."),
    _regla_rf("RF-ARR-002", Concepto.ARRENDAMIENTO_VIVIENDA, "no declarante", "0.035", "10", _INICIO_2026, _FIN_2026, "Art. 392 ET; Art. 387 ET; UVT 2026."),
    # Arrendamiento de bienes muebles: 4%, sin mínimo.
    _regla_rf("RF-ARRM-001", Concepto.ARRENDAMIENTO_MUEBLES, "declarante", "0.04", "0", _INICIO_2026, _FIN_2026, "Art. 392 ET; UVT 2026."),
    _regla_rf("RF-ARRM-002", Concepto.ARRENDAMIENTO_MUEBLES, "no declarante", "0.04", "0", _INICIO_2026, _FIN_2026, "Art. 392 ET; UVT 2026."),
    # Compras generales: 2,5% declarante / 3,5% no declarante; mínimo 10 UVT.
    _regla_rf("RF-COMP-001", Concepto.COMPRA_BIENES, "declarante", "0.025", "10", _INICIO_2026, _FIN_2026, "Art. 392 ET; Art. 401 ET; UVT 2026."),
    _regla_rf("RF-COMP-002", Concepto.COMPRA_BIENES, "no declarante", "0.035", "10", _INICIO_2026, _FIN_2026, "Art. 392 ET; Art. 401 ET; UVT 2026."),
    # Conceptos adicionales de la tabla de referencia.
    _regla_rf("RF-TRAN-001", Concepto.TRANSPORTE_CARGA, "declarante", "0.01", "2", _INICIO_2026, _FIN_2026, "Art. 392 ET; UVT 2026."),
    _regla_rf("RF-TRAN-002", Concepto.TRANSPORTE_CARGA, "no declarante", "0.01", "2", _INICIO_2026, _FIN_2026, "Art. 392 ET; UVT 2026."),
    _regla_rf("RF-HOT-001", Concepto.HOTELES_RESTAURANTES, "declarante", "0.035", "2", _INICIO_2026, _FIN_2026, "Art. 392 ET; UVT 2026."),
    _regla_rf("RF-HOT-002", Concepto.HOTELES_RESTAURANTES, "no declarante", "0.035", "2", _INICIO_2026, _FIN_2026, "Art. 392 ET; UVT 2026."),
)

TABLA_ICA: tuple[ReglaICA, ...] = (
    ReglaICA(
        municipio="Bogotá",
        actividad="Servicios",
        tarifa=D("0.00966"),
        minimo_uvt=D("0"),
        fundamento="Acuerdo 724 de 2026 (tarifa pedagógica; mínimo no configurado; verificar actividad y vigencia).",
        vigencia_desde=_INICIO_2026,
        vigencia_hasta=_FIN_2026,
    ),
)


def buscar_regla_retefuente(
    concepto: Concepto,
    declarante: bool,
    fecha: date,
    tabla: Sequence[ReglaRetefuente] = TABLA_RETEFUENTE,
) -> ReglaRetefuente:
    condicion = "declarante" if declarante else "no declarante"
    candidatas = [
        regla
        for regla in tabla
        if regla.concepto == concepto and regla.condicion == condicion and regla.vigente(fecha)
    ]
    if not candidatas:
        raise ReglaNoEncontrada(
            f"No hay regla de retefuente vigente para {concepto.value!r} "
            f"({condicion}) el {fecha.isoformat()}."
        )
    return max(candidatas, key=lambda r: r.vigencia_desde)


def buscar_regla_ica(
    municipio: str,
    actividad: str,
    fecha: date,
    tabla: Sequence[ReglaICA] = TABLA_ICA,
) -> ReglaICA | None:
    municipio_n = normalizar(municipio)
    actividad_n = normalizar(actividad)
    candidatas = [
        regla
        for regla in tabla
        if normalizar(regla.municipio) == municipio_n
        and normalizar(regla.actividad) == actividad_n
        and regla.vigente(fecha)
    ]
    if not candidatas:
        return None
    return max(candidatas, key=lambda r: r.vigencia_desde)


def categoria_reteiva(concepto: Concepto) -> str:
    return "compra" if concepto == Concepto.COMPRA_BIENES else "servicio"


def categoria_ica(concepto: Concepto) -> str:
    return "Compras" if concepto == Concepto.COMPRA_BIENES else "Servicios"


def minimo_reteiva_uvt(concepto: Concepto) -> Decimal:
    return MIN_RETEIVA_COMPRAS_UVT if categoria_reteiva(concepto) == "compra" else MIN_RETEIVA_SERVICIOS_UVT


if pd is not None:
    mostrar(
        pd.DataFrame(
            [
                {
                    "ID": regla.id,
                    "Concepto": regla.concepto.value,
                    "Condición": regla.condicion,
                    "Tarifa": porcentaje(regla.tarifa),
                    "Mínimo UVT": regla.minimo_uvt,
                    "Desde": regla.vigencia_desde,
                    "Hasta": regla.vigencia_hasta,
                }
                for regla in TABLA_RETEFUENTE
            ]
        )
    )
    mostrar(
        pd.DataFrame(
            [
                {
                    "Municipio": regla.municipio,
                    "Actividad": regla.actividad,
                    "Tarifa": porcentaje(regla.tarifa),
                    "Mínimo UVT": regla.minimo_uvt,
                    "Fundamento": regla.fundamento,
                }
                for regla in TABLA_ICA
            ]
        )
    )

,ID,Concepto,Condición,Tarifa,Mínimo UVT,Desde,Hasta
0,RF-SERV-001,Servicio general,declarante,4%,2,2026-01-01,2026-05-07
1,RF-SERV-002,Servicio general,no declarante,6%,2,2026-01-01,2026-05-07
2,RF-SERV-003,Servicio general,declarante,4%,4,2026-05-08,2026-06-30
3,RF-SERV-004,Servicio general,no declarante,6%,4,2026-05-08,2026-06-30
4,RF-SERV-005,Servicio general,declarante,4%,2,2026-07-01,2026-12-31
5,RF-SERV-006,Servicio general,no declarante,6%,2,2026-07-01,2026-12-31
6,RF-HON-001,Honorarios,declarante,11%,0,2026-01-01,2026-12-31
7,RF-HON-002,Honorarios,no declarante,10%,0,2026-01-01,2026-12-31
8,RF-ARR-001,Arrendamiento de vivienda,declarante,"3,5%",10,2026-01-01,2026-12-31
9,RF-ARR-002,Arrendamiento de vivienda,no declarante,"3,5%",10,2026-01-01,2026-12-31


,Municipio,Actividad,Tarifa,Mínimo UVT,Fundamento
0,Bogotá,Servicios,"0,966%",0,Acuerdo 724 de 2026 (tarifa pedagógica; m...


### 3.1 Tabla de decisión

Es la tabla de la sección 5 del diseño. El motor calcula con reglas, pero
conserva esta tabla para explicar y auditar la fila que corresponde al caso.

In [4]:
def _fila(
    numero: int,
    comprador: str,
    proveedor: str,
    concepto: Concepto,
    aplica_rf: bool,
    tarifa_rf: str | None,
    aplica_reteiva: str,
    aplica_reteica: str,
) -> FilaDecision:
    return FilaDecision(
        numero=numero,
        comprador=comprador,
        proveedor=proveedor,
        concepto=concepto,
        aplica_rf=aplica_rf,
        tarifa_rf=D(tarifa_rf) if tarifa_rf is not None else None,
        aplica_reteiva=aplica_reteiva,
        aplica_reteica=aplica_reteica,
    )


TABLA_DECISION: tuple[FilaDecision, ...] = (
    _fila(1, "Agente RF, no reteIVA, no reteICA", "Declarante", Concepto.SERVICIO_GENERAL, True, "0.04", "No", "No"),
    _fila(2, "Agente RF, no reteIVA, no reteICA", "No declarante", Concepto.SERVICIO_GENERAL, True, "0.06", "No", "No"),
    _fila(3, "Agente RF, no reteIVA, no reteICA", "Declarante", Concepto.HONORARIOS, True, "0.11", "No", "No"),
    _fila(4, "Agente RF, no reteIVA, no reteICA", "No declarante", Concepto.HONORARIOS, True, "0.10", "No", "No"),
    _fila(5, "Agente RF, no reteIVA, no reteICA", "Declarante", Concepto.ARRENDAMIENTO_VIVIENDA, True, "0.035", "No", "No"),
    _fila(6, "Agente RF, no reteIVA, no reteICA", "Declarante", Concepto.COMPRA_BIENES, True, "0.025", "No", "No"),
    _fila(7, "Agente RF, no reteIVA, no reteICA", "No declarante", Concepto.COMPRA_BIENES, True, "0.035", "No", "No"),
    _fila(8, "Agente RF y reteIVA", "Declarante", Concepto.SERVICIO_GENERAL, True, "0.04", "15%", "No"),
    _fila(9, "Agente RF y reteIVA", "Declarante", Concepto.COMPRA_BIENES, True, "0.025", "15%", "No"),
    _fila(10, "Agente RF y reteICA", "Declarante", Concepto.SERVICIO_GENERAL, True, "0.04", "No", "Tarifa municipal"),
    _fila(11, "Agente RF, reteIVA y reteICA", "Declarante", Concepto.SERVICIO_GENERAL, True, "0.04", "15%", "Tarifa municipal"),
    _fila(12, "Agente RF", "SIMPLE", Concepto.SERVICIO_GENERAL, False, "0", "15% si es responsable de IVA", "0%"),
    _fila(13, "Agente RF", "Autorretenedor", Concepto.SERVICIO_GENERAL, False, "0", "15% si aplica", "Según municipio"),
    _fila(14, "No agente RF", "Cualquiera", Concepto.SERVICIO_GENERAL, False, "0", "Según agente", "Según agente"),
)


def buscar_fila_decision(
    comprador: Comprador,
    proveedor: Proveedor,
    concepto: Concepto,
) -> FilaDecision | None:
    """Devuelve la fila pedagógica que mejor describe la combinación.

    La tabla es pedagógica y no cubre todas las combinaciones (por ejemplo,
    honorarios con comprador agente de reteIVA). Si no hay coincidencia, el
    motor sigue calculando con las reglas y deja la fila en ``None``.
    """
    if not comprador.agente_retefuente:
        return next(f for f in TABLA_DECISION if f.numero == 14)
    if proveedor.simple:
        return next(f for f in TABLA_DECISION if f.numero == 12)
    if proveedor.autorretenedor:
        return next(f for f in TABLA_DECISION if f.numero == 13)

    if concepto == Concepto.SERVICIO_GENERAL:
        if comprador.agente_reteiva and comprador.agente_reteica:
            return next(f for f in TABLA_DECISION if f.numero == 11)
        if comprador.agente_reteiva:
            return next(f for f in TABLA_DECISION if f.numero == 8)
        if comprador.agente_reteica:
            return next(f for f in TABLA_DECISION if f.numero == 10)
        return next(f for f in TABLA_DECISION if f.numero == (1 if proveedor.declarante else 2))

    if concepto == Concepto.HONORARIOS:
        if comprador.agente_reteiva or comprador.agente_reteica:
            return None
        return next(f for f in TABLA_DECISION if f.numero == (3 if proveedor.declarante else 4))
    if concepto == Concepto.ARRENDAMIENTO_VIVIENDA:
        if comprador.agente_reteica:
            return None
        return next(f for f in TABLA_DECISION if f.numero == 5)
    if concepto == Concepto.COMPRA_BIENES:
        if comprador.agente_reteica:
            return None
        if comprador.agente_reteiva:
            return next(f for f in TABLA_DECISION if f.numero == 9)
        return next(f for f in TABLA_DECISION if f.numero == (6 if proveedor.declarante else 7))
    return None


if pd is not None:
    mostrar(
        pd.DataFrame(
            [
                {
                    "#": fila.numero,
                    "Comprador": fila.comprador,
                    "Proveedor": fila.proveedor,
                    "Concepto": fila.concepto.value,
                    "¿RF?": "Sí" if fila.aplica_rf else "No",
                    "Tarifa RF": porcentaje(fila.tarifa_rf) if fila.tarifa_rf is not None else "—",
                    "ReteIVA": fila.aplica_reteiva,
                    "ReteICA": fila.aplica_reteica,
                }
                for fila in TABLA_DECISION
            ]
        )
    )

,#,Comprador,Proveedor,Concepto,¿RF?,Tarifa RF,ReteIVA,ReteICA
0,1,"Agente RF, no reteIVA, no reteICA",Declarante,Servicio general,Sí,4%,No,No
1,2,"Agente RF, no reteIVA, no reteICA",No declarante,Servicio general,Sí,6%,No,No
2,3,"Agente RF, no reteIVA, no reteICA",Declarante,Honorarios,Sí,11%,No,No
3,4,"Agente RF, no reteIVA, no reteICA",No declarante,Honorarios,Sí,10%,No,No
4,5,"Agente RF, no reteIVA, no reteICA",Declarante,Arrendamiento de vivienda,Sí,"3,5%",No,No
5,6,"Agente RF, no reteIVA, no reteICA",Declarante,Compra de bienes,Sí,"2,5%",No,No
6,7,"Agente RF, no reteIVA, no reteICA",No declarante,Compra de bienes,Sí,"3,5%",No,No
7,8,Agente RF y reteIVA,Declarante,Servicio general,Sí,4%,15%,No
8,9,Agente RF y reteIVA,Declarante,Compra de bienes,Sí,"2,5%",15%,No
9,10,Agente RF y reteICA,Declarante,Servicio general,Sí,4%,No,Tarifa municipal


### 3.2 Tabla de cuentas contables

In [5]:
CUENTAS = {
    "servicios": ("5135", "Servicios"),
    "honorarios": ("5110", "Honorarios"),
    "arrendamientos": ("5120", "Arrendamientos"),
    "inventarios": ("143505", "Inventarios"),
    "iva_descontable": ("240805", "IVA descontable"),
    "proveedores": ("220505", "Proveedores nacionales"),
    "bancos": ("111005", "Bancos"),
    "rf_servicios": ("236525", "Retención en la fuente — servicios"),
    "rf_honorarios": ("236515", "Retención en la fuente — honorarios"),
    "rf_arrendamientos": ("236530", "Retención en la fuente — arrendamientos"),
    "rf_compras": ("236540", "Retención en la fuente — compras"),
    "reteiva": ("236701", "Impuesto a las ventas retenido"),
    "reteica": ("2368", "Impuesto de industria y comercio retenido"),
}


def cuenta_gasto(concepto: Concepto) -> tuple[str, str]:
    if concepto == Concepto.HONORARIOS:
        return CUENTAS["honorarios"]
    if concepto in (Concepto.ARRENDAMIENTO_VIVIENDA, Concepto.ARRENDAMIENTO_MUEBLES):
        return CUENTAS["arrendamientos"]
    if concepto == Concepto.COMPRA_BIENES:
        return CUENTAS["inventarios"]
    return CUENTAS["servicios"]


def cuenta_retefuente(concepto: Concepto) -> tuple[str, str]:
    if concepto == Concepto.HONORARIOS:
        return CUENTAS["rf_honorarios"]
    if concepto in (Concepto.ARRENDAMIENTO_VIVIENDA, Concepto.ARRENDAMIENTO_MUEBLES):
        return CUENTAS["rf_arrendamientos"]
    if concepto == Concepto.COMPRA_BIENES:
        return CUENTAS["rf_compras"]
    return CUENTAS["rf_servicios"]

## 4. Normalización de la factura

Reglas de la sección 6:
- IVA = base × tarifa.
- Si el total incluye IVA: base = total / (1 + tarifa).
- La base de retefuente nunca incluye IVA.
- La moneda extranjera se convierte con la TRM, sin diferencia en cambio.

In [6]:
def normalizar_factura(operacion: Operacion) -> DatosFactura:
    moneda = operacion.moneda.upper().strip()
    if moneda == "COP":
        factor = D("1")
        trm = D("1")
    elif moneda == "USD":
        if operacion.trm is None or D(operacion.trm) <= 0:
            raise CalculoBloqueado("Si la moneda es USD, la TRM debe ser mayor que cero.")
        trm = D(operacion.trm)
        factor = trm
    else:
        raise CalculoBloqueado(f"Moneda no soportada: {operacion.moneda!r}. Use COP o USD.")

    tarifa = D(operacion.tarifa_iva)
    if operacion.iva_excluido and tarifa != IVA_EXCLUIDO:
        raise CalculoBloqueado("La operación está marcada como IVA excluido, pero la tarifa no es 0%.")

    base_ingresada = D(operacion.base) * factor if operacion.base is not None else None
    iva_ingresado = D(operacion.iva) * factor if operacion.iva is not None else None
    total_ingresado = D(operacion.total) * factor if operacion.total is not None else None
    descuentos = D(operacion.descuentos) * factor

    if descuentos < 0:
        raise CalculoBloqueado("Los descuentos no pueden ser negativos.")

    if operacion.total_incluye_iva:
        if total_ingresado is None:
            raise CalculoBloqueado("Si el total incluye IVA, debe informar el total.")
        if total_ingresado <= 0:
            raise CalculoBloqueado("El total debe ser mayor que cero.")
        base = dinero(total_ingresado / (D("1") + tarifa))
        iva = dinero(total_ingresado - base)
        if base_ingresada is not None:
            base_declarada = base_ingresada - descuentos
            if abs(base_declarada - base) > CENTAVO:
                raise CalculoBloqueado(
                    "La base informada no coincide con la base implícita en el total con IVA."
                )
        if iva_ingresado is not None and abs(iva_ingresado - iva) > CENTAVO:
            raise CalculoBloqueado(
                "El IVA informado no coincide con el IVA implícito en el total con IVA."
            )
        base_bruta = dinero(base + descuentos)
        total = dinero(total_ingresado)
    else:
        if base_ingresada is None:
            if total_ingresado is not None and iva_ingresado is not None:
                base = dinero(total_ingresado - iva_ingresado)
            elif total_ingresado is not None and tarifa == IVA_EXCLUIDO:
                base = dinero(total_ingresado)
            else:
                raise CalculoBloqueado(
                    "Falta la base. Informe base, o total e IVA, para poder liquidar."
                )
            base_bruta = dinero(base + descuentos)
        else:
            base = dinero(base_ingresada - descuentos)
            base_bruta = dinero(base_ingresada)

        if base <= 0:
            raise CalculoBloqueado("La base después de descuentos debe ser mayor que cero.")

        iva = dinero(iva_ingresado) if iva_ingresado is not None else dinero(base * tarifa)
        total = dinero(total_ingresado) if total_ingresado is not None else dinero(base + iva)

        if iva < 0:
            raise CalculoBloqueado("El IVA no puede ser negativo.")
        if abs(dinero(base + iva) - total) > CENTAVO:
            raise CalculoBloqueado(
                f"El total no cuadra: base {cop(base)} + IVA {cop(iva)} = {cop(base + iva)}, "
                f"pero el total informado es {cop(total)}."
            )

    if base <= 0:
        raise CalculoBloqueado("La base debe ser mayor que cero.")
    if descuentos > base_bruta:
        raise CalculoBloqueado("Los descuentos no pueden superar la base bruta.")

    return DatosFactura(
        base_bruta=dinero(base_bruta),
        descuentos=dinero(descuentos),
        base=dinero(base),
        iva=dinero(iva),
        total=dinero(total),
        tarifa_iva=tarifa,
        moneda_original=moneda,
        trm=trm,
        factor_conversion=factor,
    )

## 5. Validaciones y bloqueos

El principio del diseño es: **si no se puede explicar, no se automatiza**.
Las validaciones críticas lanzan ``CalculoBloqueado``; las advertencias se
guardan en la liquidación y se muestran al usuario.

In [7]:
def _alerta(codigo: str, mensaje: str, severidad: Severidad = Severidad.ALERTA, campo: str | None = None) -> Alerta:
    return Alerta(codigo=codigo, mensaje=mensaje, severidad=severidad, campo=campo)


def _exigir_sin_bloqueos(alertas: Sequence[Alerta]) -> None:
    bloqueos = [a for a in alertas if a.severidad == Severidad.BLOQUEO]
    if bloqueos:
        raise CalculoBloqueado(
            "No se puede liquidar: " + " | ".join(a.mensaje for a in bloqueos),
            alertas,
        )


def validar_entradas(comprador: Comprador, proveedor: Proveedor, operacion: Operacion) -> list[Alerta]:
    alertas: list[Alerta] = []

    if not comprador.nit.strip():
        alertas.append(_alerta("COMP-NIT", "El comprador no tiene NIT.", Severidad.BLOQUEO, "comprador.nit"))
    if not comprador.razon_social.strip():
        alertas.append(_alerta("COMP-NOMBRE", "El comprador no tiene razón social.", Severidad.BLOQUEO, "comprador.razon_social"))
    if not proveedor.nit.strip():
        alertas.append(_alerta("PROV-NIT", "El proveedor no está identificado: falta el NIT.", Severidad.BLOQUEO, "proveedor.nit"))
    if not proveedor.razon_social.strip():
        alertas.append(_alerta("PROV-NOMBRE", "El proveedor no tiene razón social.", Severidad.BLOQUEO, "proveedor.razon_social"))
    if not operacion.numero.strip():
        alertas.append(_alerta("OP-NUMERO", "La operación no tiene número de documento.", Severidad.BLOQUEO, "operacion.numero"))
    if operacion.fecha_pago < operacion.fecha_emision:
        alertas.append(_alerta("OP-FECHAS", "La fecha de pago no puede ser anterior a la fecha de emisión.", Severidad.BLOQUEO, "operacion.fecha_pago"))
    if operacion.fecha_emision.year != operacion.fecha_pago.year:
        alertas.append(_alerta("OP-ANIO", "La operación cruza año: revise la UVT y la vigencia aplicable.", Severidad.BLOQUEO, "operacion.fecha_pago"))
    if D(operacion.tarifa_iva) not in TARIFAS_IVA_PERMITIDAS:
        alertas.append(_alerta("IVA-TARIFA", f"Tarifa de IVA no permitida: {porcentaje(operacion.tarifa_iva)}.", Severidad.BLOQUEO, "operacion.tarifa_iva"))
    if operacion.tipo_documento == TipoDocumento.NOTA_CREDITO:
        alertas.append(_alerta("OP-NC", "La nota crédito se procesa con el módulo de ajustes, no como factura.", Severidad.BLOQUEO, "operacion.tipo_documento"))

    if not proveedor.residente:
        alertas.append(_alerta("PROV-EXTERIOR", "Proveedor no residente: el caso se deriva al módulo de pagos al exterior (fuera de la v1).", Severidad.BLOQUEO, "proveedor.residente"))
    if not comprador.obligado_contabilidad:
        alertas.append(_alerta("COMP-CONTABILIDAD", "El comprador no está obligado a llevar contabilidad; la v1 no cubre ese caso.", Severidad.BLOQUEO, "comprador.obligado_contabilidad"))
    if comprador.regimen == Regimen.SIMPLE:
        alertas.append(_alerta("COMP-SIMPLE", "La v1 no cubre compradores del régimen SIMPLE.", Severidad.BLOQUEO, "comprador.regimen"))
    if comprador.gran_contribuyente and proveedor.gran_contribuyente:
        alertas.append(_alerta("RETEIVA-GC", "Operación entre grandes contribuyentes: posible exclusión de reteIVA; requiere revisión humana.", Severidad.BLOQUEO, "proveedor.gran_contribuyente"))
    if proveedor.autorretencion_especial and not proveedor.autorretenedor:
        alertas.append(_alerta("AUTORRET-ESPECIAL", "El proveedor solo tiene autorretención especial (código 59). No se asume exención de la retención general: revisión humana.", Severidad.BLOQUEO, "proveedor.autorretencion_especial"))
    if proveedor.tipo == TipoPersona.JURIDICA and not proveedor.declarante and not proveedor.simple:
        alertas.append(_alerta("PROV-DECLARANTE", "Una persona jurídica debe ser declarante de renta; revise el RUT del proveedor.", Severidad.BLOQUEO, "proveedor.declarante"))
    if not proveedor.declarante and not proveedor.simple and not proveedor.soporte_no_declarante:
        alertas.append(_alerta("PROV-SOPORTE", "No hay certificación de no declarante. Use la tarifa conservadora o bloquee; la v1 bloquea.", Severidad.BLOQUEO, "proveedor.soporte_no_declarante"))
    if not proveedor.rut_recibido:
        alertas.append(_alerta("PROV-RUT", "Falta el RUT del proveedor: no se puede autorizar el pago.", Severidad.ALERTA, "proveedor.rut_recibido"))
    if not operacion.factura_validada:
        alertas.append(_alerta("DOC-FACTURA", "La factura no está validada por la DIAN: se bloquea el IVA y el costo.", Severidad.BLOQUEO, "operacion.factura_validada"))
    if operacion.periodo_cerrado:
        alertas.append(_alerta("PERIODO-CERRADO", "El periodo está cerrado; se requiere un ajuste autorizado.", Severidad.BLOQUEO, "operacion.periodo_cerrado"))
    if operacion.retencion_ya_practicada:
        alertas.append(_alerta("RETENCION-DUP", "La retención ya fue practicada; no se permite un segundo cálculo sin autorización.", Severidad.BLOQUEO, "operacion.retencion_ya_practicada"))
    if not operacion.descripcion.strip():
        alertas.append(_alerta("OP-DESCRIPCION", "Falta la descripción de la operación.", Severidad.ALERTA, "operacion.descripcion"))
    if comprador.municipio.strip() == "" and comprador.agente_reteica:
        alertas.append(_alerta("COMP-MUNICIPIO", "El comprador es agente de reteICA pero no tiene municipio.", Severidad.BLOQUEO, "comprador.municipio"))

    return alertas


def _validar_valor_pagado(operacion: Operacion, total: Decimal) -> None:
    if operacion.valor_pagado is None:
        return
    valor = D(operacion.valor_pagado)
    if valor <= 0:
        raise CalculoBloqueado("El valor pagado debe ser mayor que cero.")
    if valor > total + CENTAVO:
        raise CalculoBloqueado("El valor pagado no puede superar el total de la factura.")

## 6. Motor de liquidación

In [8]:
class RepositorioFacturas:
    """Memoria mínima para detectar duplicados por proveedor + número."""

    def __init__(self) -> None:
        self._facturas: set[tuple[str, str]] = set()

    def existe(self, proveedor_nit: str, numero: str) -> bool:
        return (normalizar(proveedor_nit), normalizar(numero)) in self._facturas

    def registrar(self, proveedor_nit: str, numero: str) -> None:
        self._facturas.add((normalizar(proveedor_nit), normalizar(numero)))


class LiquidadorRetenciones:
    """Motor que aplica las reglas del diseño funcional."""

    def __init__(
        self,
        usuario: str = "usuario.demo",
        repositorio: RepositorioFacturas | None = None,
        tabla_uvt: dict[int, Decimal] | None = None,
        tabla_retefuente: Sequence[ReglaRetefuente] | None = None,
        tabla_ica: Sequence[ReglaICA] | None = None,
    ) -> None:
        self.usuario = usuario
        self.repositorio = repositorio
        self.tabla_uvt = tabla_uvt or TABLA_UVT
        self.tabla_retefuente = tuple(tabla_retefuente or TABLA_RETEFUENTE)
        self.tabla_ica = tuple(tabla_ica or TABLA_ICA)

    def _uvt(self, fecha: date) -> tuple[Decimal, int]:
        if fecha.year not in self.tabla_uvt:
            raise ReglaNoEncontrada(f"No hay UVT cargada para el año {fecha.year}.")
        return self.tabla_uvt[fecha.year], fecha.year

    def calcular(
        self,
        comprador: Comprador,
        proveedor: Proveedor,
        operacion: Operacion,
        autorizar_duplicado: bool = False,
    ) -> Liquidacion:
        alertas = validar_entradas(comprador, proveedor, operacion)
        _exigir_sin_bloqueos(alertas)

        if self.repositorio and self.repositorio.existe(proveedor.nit, operacion.numero) and not autorizar_duplicado:
            raise CalculoBloqueado(
                f"Posible factura duplicada: ya existe {operacion.numero} del proveedor {proveedor.nit}."
            )

        datos = normalizar_factura(operacion)
        if datos.moneda_original == "USD":
            alertas.append(
                _alerta(
                    "MONEDA-USD",
                    f"Conversión simple a COP con TRM {datos.trm}. No incluye diferencia en cambio.",
                    Severidad.INFO,
                    "operacion.trm",
                )
            )

        uvt, uvt_anio = self._uvt(operacion.fecha_emision)
        _validar_valor_pagado(operacion, datos.total)

        # Pago total o parcial. Las retenciones se calculan sobre la parte pagada.
        if operacion.valor_pagado is None:
            factor_pago = D("1")
            total_pago = datos.total
        else:
            total_pago = dinero(operacion.valor_pagado)
            factor_pago = total_pago / datos.total
            if factor_pago < 1:
                alertas.append(
                    _alerta(
                        "PAGO-PARCIAL",
                        "Pago parcial: las retenciones se calculan sobre la parte pagada, no sobre la factura completa.",
                        Severidad.INFO,
                        "operacion.valor_pagado",
                    )
                )

        base_pago = dinero(datos.base * factor_pago)
        iva_pago = dinero(datos.iva * factor_pago)

        # --- Retefuente (renta) ---
        regla_rf: ReglaRetefuente | None = None
        tarifa_rf = CERO
        base_rf = base_pago
        retefuente = CERO
        minimo_rf_uvt = CERO
        minimo_rf_valor = CERO
        cumple_minimo_rf = True

        if not comprador.agente_retefuente:
            alertas.append(_alerta("RF-AGENTE", "El comprador no es agente de retefuente; no se practica retefuente.", Severidad.INFO))
        elif proveedor.simple:
            alertas.append(_alerta("RF-SIMPLE", "Proveedor SIMPLE: no se practica retefuente (Art. 911 ET).", Severidad.INFO))
        elif proveedor.autorretenedor:
            alertas.append(_alerta("RF-AUTORRET", "Proveedor autorretenedor: el comprador no practica retefuente.", Severidad.INFO))
        else:
            regla_rf = buscar_regla_retefuente(
                operacion.concepto,
                proveedor.declarante,
                operacion.fecha_emision,
                self.tabla_retefuente,
            )
            minimo_rf_uvt = regla_rf.minimo_uvt
            minimo_rf_valor = dinero(minimo_rf_uvt * uvt)
            cumple_minimo_rf = base_pago >= minimo_rf_valor
            if not cumple_minimo_rf:
                alertas.append(
                    _alerta(
                        "RF-MINIMO",
                        f"La base {cop(base_pago)} es inferior al mínimo de {minimo_rf_uvt} UVT "
                        f"({cop(minimo_rf_valor)}); no se practica retefuente.",
                        Severidad.INFO,
                    )
                )
            else:
                tarifa_rf = regla_rf.tarifa
                retefuente = dinero(base_pago * tarifa_rf)

        # --- ReteIVA ---
        aplica_reteiva = False
        tarifa_reteiva = TARIFA_RETEIVA
        base_reteiva = iva_pago
        reteiva = CERO
        min_riva_uvt = minimo_reteiva_uvt(operacion.concepto)
        min_riva_valor = dinero(min_riva_uvt * uvt)
        cumple_minimo_reteiva = iva_pago > 0 and base_pago >= min_riva_valor

        if not comprador.agente_reteiva:
            alertas.append(_alerta("RIVA-AGENTE", "El comprador no es agente de reteIVA; no se practica reteIVA.", Severidad.INFO))
        elif not proveedor.responsable_iva:
            alertas.append(_alerta("RIVA-RESPONSABLE", "El proveedor no es responsable de IVA; no se practica reteIVA.", Severidad.INFO))
        elif iva_pago <= 0:
            alertas.append(_alerta("RIVA-SIN-IVA", "La operación no tiene IVA; no se practica reteIVA.", Severidad.INFO))
        elif not cumple_minimo_reteiva:
            alertas.append(
                _alerta(
                    "RIVA-MINIMO",
                    f"La base {cop(base_pago)} no supera el mínimo de reteIVA de {min_riva_uvt} UVT "
                    f"({cop(min_riva_valor)}); no se practica reteIVA.",
                    Severidad.INFO,
                )
            )
        else:
            aplica_reteiva = True
            reteiva = dinero(iva_pago * tarifa_reteiva)

        # --- ReteICA ---
        aplica_reteica = False
        regla_ica: ReglaICA | None = None
        tarifa_reteica = CERO
        base_reteica = base_pago
        reteica = CERO
        minimo_reteica_uvt = CERO
        minimo_reteica_valor = CERO
        cumple_minimo_reteica = True

        if not comprador.agente_reteica:
            alertas.append(_alerta("RICA-AGENTE", "El comprador no es agente de reteICA; no se practica reteICA.", Severidad.INFO))
        elif proveedor.simple:
            alertas.append(_alerta("RICA-SIMPLE", "Proveedor SIMPLE: no se practica reteICA.", Severidad.INFO))
        elif not proveedor.actividad_gravada_ica:
            alertas.append(_alerta("RICA-ACTIVIDAD", "La actividad del proveedor no está gravada en ICA; no se practica reteICA.", Severidad.INFO))
        else:
            regla_ica = buscar_regla_ica(
                comprador.municipio,
                categoria_ica(operacion.concepto),
                operacion.fecha_emision,
                self.tabla_ica,
            )
            if regla_ica is None:
                raise ReglaNoEncontrada(
                    f"No hay tarifa de reteICA para {comprador.municipio!r}, "
                    f"actividad {categoria_ica(operacion.concepto)!r}, el {operacion.fecha_emision.isoformat()}. "
                    "No se inventa la tarifa."
                )
            minimo_reteica_uvt = regla_ica.minimo_uvt
            minimo_reteica_valor = dinero(minimo_reteica_uvt * uvt)
            cumple_minimo_reteica = base_pago >= minimo_reteica_valor
            if not cumple_minimo_reteica:
                alertas.append(
                    _alerta(
                        "RICA-MINIMO",
                        f"La base {cop(base_pago)} es inferior al mínimo municipal de "
                        f"{minimo_reteica_uvt} UVT ({cop(minimo_reteica_valor)}); no se practica reteICA.",
                        Severidad.INFO,
                    )
                )
            else:
                aplica_reteica = True
                tarifa_reteica = regla_ica.tarifa
                reteica = dinero(base_pago * tarifa_reteica)

        total_retenciones = dinero(retefuente + reteiva + reteica)
        neto = dinero(total_pago - total_retenciones)
        if neto < 0:
            raise CalculoBloqueado(
                f"El neto a pagar sería negativo ({cop(neto)}). Revise las retenciones y el valor pagado."
            )

        if not proveedor.declarante and not proveedor.simple and proveedor.rut_recibido:
            alertas.append(_alerta("PROV-NO-DECL", "Proveedor no declarante: conserve la certificación y el RUT en el expediente.", Severidad.INFO))

        fila_decision = buscar_fila_decision(comprador, proveedor, operacion.concepto)
        if fila_decision is None:
            alertas.append(_alerta("DECISION-NO-TABULADA", "La combinación no está en la tabla pedagógica; se aplicaron las reglas por concepto.", Severidad.INFO))

        liquidacion = Liquidacion(
            comprador=comprador,
            proveedor=proveedor,
            operacion=operacion,
            datos=datos,
            uvt=uvt,
            uvt_anio=uvt_anio,
            factor_pago=factor_pago,
            base_pago=base_pago,
            iva_pago=iva_pago,
            total_pago=total_pago,
            regla_rf=regla_rf,
            tarifa_rf=tarifa_rf,
            base_rf=base_rf,
            retefuente=retefuente,
            minimo_rf_uvt=minimo_rf_uvt,
            minimo_rf_valor=minimo_rf_valor,
            cumple_minimo_rf=cumple_minimo_rf,
            aplica_reteiva=aplica_reteiva,
            tarifa_reteiva=tarifa_reteiva,
            base_reteiva=base_reteiva,
            reteiva=reteiva,
            minimo_reteiva_uvt=min_riva_uvt,
            minimo_reteiva_valor=min_riva_valor,
            cumple_minimo_reteiva=cumple_minimo_reteiva,
            regla_ica=regla_ica,
            aplica_reteica=aplica_reteica,
            tarifa_reteica=tarifa_reteica,
            base_reteica=base_reteica,
            reteica=reteica,
            minimo_reteica_uvt=minimo_reteica_uvt,
            minimo_reteica_valor=minimo_reteica_valor,
            cumple_minimo_reteica=cumple_minimo_reteica,
            total_retenciones=total_retenciones,
            neto=neto,
            fila_decision=fila_decision,
            alertas=alertas,
        )

        liquidacion.asientos = self._crear_asientos(liquidacion)
        liquidacion.certificados = self._crear_certificados(liquidacion)
        liquidacion.checklist = self._crear_checklist(liquidacion)
        liquidacion.auditoria = self._crear_auditoria(liquidacion)

        if not liquidacion.asientos_cuadran:
            raise CalculoBloqueado("El asiento no cuadra: débitos ≠ créditos. No se permite guardar.")

        if self.repositorio:
            self.repositorio.registrar(proveedor.nit, operacion.numero)

        return liquidacion

    # ------------------------------------------------------------------
    # Asientos
    # ------------------------------------------------------------------
    def _crear_asientos(self, liq: Liquidacion) -> list[Asiento]:
        op = liq.operacion
        tercero = f"{liq.proveedor.razon_social} / {liq.proveedor.nit}"
        documento = op.numero
        asientos: list[Asiento] = []

        # 1) Causación: gasto/inventario + IVA descontable contra proveedores.
        cuenta, nombre_cuenta = cuenta_gasto(op.concepto)
        lineas_causacion: list[LineaAsiento] = []
        if liq.comprador.responsable_iva and liq.datos.iva > 0:
            lineas_causacion.append(
                LineaAsiento(cuenta, nombre_cuenta, liq.datos.base, CERO, tercero, op.centro_costo, op.descripcion, documento)
            )
            lineas_causacion.append(
                LineaAsiento(
                    CUENTAS["iva_descontable"][0],
                    CUENTAS["iva_descontable"][1],
                    liq.datos.iva,
                    CERO,
                    tercero,
                    op.centro_costo,
                    "IVA descontable de la factura",
                    documento,
                )
            )
        else:
            lineas_causacion.append(
                LineaAsiento(cuenta, nombre_cuenta, liq.datos.total, CERO, tercero, op.centro_costo, op.descripcion, documento)
            )
        lineas_causacion.append(
            LineaAsiento(
                CUENTAS["proveedores"][0],
                CUENTAS["proveedores"][1],
                CERO,
                liq.datos.total,
                tercero,
                op.centro_costo,
                "Proveedor nacional",
                documento,
            )
        )
        asientos.append(Asiento("Causación", op.fecha_emision, documento, "Causación de la factura", lineas_causacion))

        # 2) Pago: proveedores contra banco y retenciones.
        lineas_pago: list[LineaAsiento] = [
            LineaAsiento(
                CUENTAS["proveedores"][0],
                CUENTAS["proveedores"][1],
                liq.total_pago,
                CERO,
                tercero,
                op.centro_costo,
                "Pago total o parcial de la factura",
                documento,
            ),
            LineaAsiento(
                CUENTAS["bancos"][0],
                CUENTAS["bancos"][1],
                CERO,
                liq.neto,
                tercero,
                op.centro_costo,
                "Salida de bancos",
                documento,
            ),
        ]
        if liq.retefuente > 0:
            cta, nom = cuenta_retefuente(op.concepto)
            lineas_pago.append(LineaAsiento(cta, nom, CERO, liq.retefuente, tercero, op.centro_costo, "Retención practicada", documento))
        if liq.reteiva > 0:
            lineas_pago.append(
                LineaAsiento(CUENTAS["reteiva"][0], CUENTAS["reteiva"][1], CERO, liq.reteiva, tercero, op.centro_costo, "ReteIVA practicada", documento)
            )
        if liq.reteica > 0:
            lineas_pago.append(
                LineaAsiento(CUENTAS["reteica"][0], CUENTAS["reteica"][1], CERO, liq.reteica, tercero, op.centro_costo, "ReteICA practicada", documento)
            )
        asientos.append(Asiento("Pago", op.fecha_pago, documento, "Pago de la factura y retenciones", lineas_pago))

        # 3) Pago de retenciones: pasivo contra banco.
        if liq.total_retenciones > 0:
            lineas_retenciones: list[LineaAsiento] = []
            if liq.retefuente > 0:
                cta, nom = cuenta_retefuente(op.concepto)
                lineas_retenciones.append(LineaAsiento(cta, nom, liq.retefuente, CERO, tercero, op.centro_costo, "Pago de retefuente", documento))
            if liq.reteiva > 0:
                lineas_retenciones.append(
                    LineaAsiento(CUENTAS["reteiva"][0], CUENTAS["reteiva"][1], liq.reteiva, CERO, tercero, op.centro_costo, "Pago de reteIVA", documento)
                )
            if liq.reteica > 0:
                lineas_retenciones.append(
                    LineaAsiento(CUENTAS["reteica"][0], CUENTAS["reteica"][1], liq.reteica, CERO, tercero, op.centro_costo, "Pago de reteICA", documento)
                )
            lineas_retenciones.append(
                LineaAsiento(
                    CUENTAS["bancos"][0],
                    CUENTAS["bancos"][1],
                    CERO,
                    liq.total_retenciones,
                    tercero,
                    op.centro_costo,
                    "Pago de retenciones",
                    documento,
                )
            )
            asientos.append(
                Asiento("Pago de retenciones", op.fecha_pago, documento, "Pago de las retenciones practicadas", lineas_retenciones)
            )

        return asientos

    # ------------------------------------------------------------------
    # Certificados
    # ------------------------------------------------------------------
    def _crear_certificados(self, liq: Liquidacion) -> list[CertificadoRetencion]:
        op = liq.operacion
        certificados: list[CertificadoRetencion] = []

        def base_cert(tipo: str, concepto: str, base: Decimal, tarifa: Decimal, valor: Decimal, fundamento: str) -> CertificadoRetencion:
            return CertificadoRetencion(
                tipo=tipo,
                numero=f"{tipo[:3].upper()}-{op.numero}-{liq.uvt_anio}",
                anio_gravable=liq.uvt_anio,
                agente_nit=liq.comprador.nit,
                agente_nombre=liq.comprador.razon_social,
                retenido_nit=liq.proveedor.nit,
                retenido_nombre=liq.proveedor.razon_social,
                concepto=concepto,
                base=base,
                tarifa=tarifa,
                valor=valor,
                fundamento=fundamento,
                ciudad=liq.comprador.municipio,
                fecha_expedicion=op.fecha_pago,
            )

        if liq.retefuente > 0 and liq.regla_rf:
            certificados.append(
                base_cert(
                    "Renta",
                    liq.regla_rf.concepto.value,
                    liq.base_rf,
                    liq.tarifa_rf,
                    liq.retefuente,
                    liq.regla_rf.fundamento,
                )
            )
        if liq.reteiva > 0:
            certificado = base_cert(
                "ReteIVA",
                f"ReteIVA sobre {op.concepto.value}",
                liq.base_reteiva,
                liq.tarifa_reteiva,
                liq.reteiva,
                "Art. 437-2 ET; Decreto 1625 de 2016, art. 1.6.1.12.13.",
            )
            certificado.base_sin_iva = liq.base_pago
            certificado.iva = liq.iva_pago
            certificados.append(certificado)
        if liq.reteica > 0 and liq.regla_ica:
            certificados.append(
                base_cert(
                    "ReteICA",
                    f"ReteICA — {liq.regla_ica.actividad}",
                    liq.base_reteica,
                    liq.tarifa_reteica,
                    liq.reteica,
                    liq.regla_ica.fundamento,
                )
            )
        for certificado in certificados:
            faltantes = certificado.validar()
            if faltantes:
                raise CalculoBloqueado(
                    f"El certificado {certificado.numero} está incompleto: {', '.join(faltantes)}."
                )
        return certificados

    # ------------------------------------------------------------------
    # Checklist documental
    # ------------------------------------------------------------------
    def _crear_checklist(self, liq: Liquidacion) -> list[ItemChecklist]:
        op = liq.operacion
        items = [
            ItemChecklist("RUT del proveedor", True, liq.proveedor.rut_recibido, "Falta el RUT." if not liq.proveedor.rut_recibido else ""),
            ItemChecklist("Factura electrónica validada", True, op.factura_validada, "" if op.factura_validada else "No validada por la DIAN."),
            ItemChecklist("Orden de compra", True, op.orden_compra, "" if op.orden_compra else "Falta la orden de compra."),
            ItemChecklist("Acta de recibido", True, op.acta_recibido, "" if op.acta_recibido else "Falta el acta de recibido."),
            ItemChecklist("Cálculo de retención", True, True, "Generado por el motor."),
            ItemChecklist("Asiento contable", True, liq.asientos_cuadran, "" if liq.asientos_cuadran else "El asiento no cuadra."),
        ]
        if liq.retefuente > 0:
            items.append(ItemChecklist("Certificado de renta", True, True, "Art. 381 ET."))
        if liq.reteiva > 0:
            items.append(ItemChecklist("Certificado de reteIVA", True, True, "Base sin IVA, IVA, tarifa y valor."))
        if liq.reteica > 0:
            items.append(ItemChecklist("Certificado de reteICA", True, True, "Requisitos municipales."))
        if liq.retefuente > 0 or liq.reteiva > 0:
            items.append(ItemChecklist("Borrador Formulario 350", True, True, "No se presenta automáticamente."))
        if liq.reteica > 0:
            items.append(ItemChecklist("Borrador declaración municipal", True, True, "No se presenta automáticamente."))
        return items

    # ------------------------------------------------------------------
    # Auditoría
    # ------------------------------------------------------------------
    def _crear_auditoria(self, liq: Liquidacion) -> RegistroAuditoria:
        entrada = {
            "comprador": asdict(liq.comprador),
            "proveedor": asdict(liq.proveedor),
            "operacion": asdict(liq.operacion),
            "uvt": liq.uvt,
            "version_tabla": VERSION_TABLAS,
        }
        entrada_json = json.dumps(entrada, default=str, ensure_ascii=False, sort_keys=True)
        resultado_json = json.dumps(liq.resumen(), default=str, ensure_ascii=False, sort_keys=True)
        regla_id = liq.regla_rf.id if liq.regla_rf else ""
        fundamento = liq.regla_rf.fundamento if liq.regla_rf else ""
        documentos = tuple(cert.numero for cert in liq.certificados) + tuple(
            item.documento for item in liq.checklist if item.obligatorio and item.recibido
        )
        return RegistroAuditoria(
            usuario=self.usuario,
            fecha_hora=datetime.now(),
            version_tabla=VERSION_TABLAS,
            uvt=liq.uvt,
            uvt_anio=liq.uvt_anio,
            regla_id=regla_id,
            fundamento=fundamento,
            hash_entrada=hashlib.sha256(entrada_json.encode("utf-8")).hexdigest(),
            entrada_json=entrada_json,
            resultado_json=resultado_json,
            documentos=documentos,
            cambios=("Creación de la liquidación",),
        )

## 7. Presentación de resultados

In [9]:
def tabla_asiento(asiento: Asiento) -> Any:
    filas = [
        {
            "Cuenta": linea.cuenta,
            "Nombre": linea.nombre,
            "Débito": cop(linea.debito) if linea.debito else "",
            "Crédito": cop(linea.credito) if linea.credito else "",
            "Tercero": linea.tercero,
            "Centro de costo": linea.centro_costo,
            "Documento": linea.documento,
        }
        for linea in asiento.lineas
    ]
    if pd is None:
        return filas
    return pd.DataFrame(filas)


def tabla_resultados(resultados: Sequence[tuple[Any, Liquidacion]]) -> Any:
    filas = []
    for caso, liq in resultados:
        filas.append(
            {
                "#": caso.numero,
                "Caso": caso.nombre,
                "Base pago": cop(liq.base_pago),
                "IVA": cop(liq.iva_pago),
                "Total pago": cop(liq.total_pago),
                "Retefuente": cop(liq.retefuente),
                "ReteIVA": cop(liq.reteiva),
                "ReteICA": cop(liq.reteica),
                "Neto": cop(liq.neto),
                "Escenario": liq.fila_decision.numero if liq.fila_decision else "—",
            }
        )
    if pd is None:
        return filas
    return pd.DataFrame(filas)


def borradores_declaracion(liq: Liquidacion) -> Any:
    """Resumen para los borradores de Formulario 350 y declaración municipal."""
    filas = []
    if liq.retefuente > 0:
        filas.append(
            {
                "Declaración": "Formulario 350",
                "Concepto": "Retefuente",
                "Base": cop(liq.base_rf),
                "Tarifa": porcentaje(liq.tarifa_rf),
                "Valor": cop(liq.retefuente),
            }
        )
    if liq.reteiva > 0:
        filas.append(
            {
                "Declaración": "Formulario 350",
                "Concepto": "ReteIVA",
                "Base": cop(liq.iva_pago),
                "Tarifa": porcentaje(liq.tarifa_reteiva),
                "Valor": cop(liq.reteiva),
            }
        )
    if liq.reteica > 0 and liq.regla_ica:
        filas.append(
            {
                "Declaración": f"Municipal — {liq.regla_ica.municipio}",
                "Concepto": "ReteICA",
                "Base": cop(liq.base_reteica),
                "Tarifa": porcentaje(liq.tarifa_reteica),
                "Valor": cop(liq.reteica),
            }
        )
    if pd is None:
        return filas
    return pd.DataFrame(filas)


def explicar(liq: Liquidacion) -> str:
    op = liq.operacion
    lineas = [
        f"Concepto: {op.concepto.value}",
        f"Base factura: {cop(liq.datos.base)}",
        f"IVA: {cop(liq.datos.iva)} ({porcentaje(liq.datos.tarifa_iva)})",
        f"Total factura: {cop(liq.datos.total)}",
    ]
    if liq.factor_pago < 1:
        lineas.append(
            f"Pago parcial: {porcentaje(liq.factor_pago)} de la factura; "
            f"base del pago {cop(liq.base_pago)}, IVA del pago {cop(liq.iva_pago)}."
        )
    if liq.regla_rf:
        lineas.extend(
            [
                f"Mínimo retefuente: {liq.minimo_rf_uvt} UVT = {cop(liq.minimo_rf_valor)} → "
                f"{'cumple' if liq.cumple_minimo_rf else 'no cumple'}",
                f"Tarifa retefuente: {porcentaje(liq.tarifa_rf)} "
                f"({'declarante' if liq.proveedor.declarante else 'no declarante'})",
                f"Retefuente: {cop(liq.retefuente)}",
                f"Fundamento: {liq.regla_rf.fundamento}",
            ]
        )
    else:
        lineas.append("Retefuente: no aplica (comprador no agente, SIMPLE o autorretenedor).")

    if liq.aplica_reteiva:
        lineas.append(
            f"ReteIVA: {cop(liq.reteiva)} = {porcentaje(liq.tarifa_reteiva)} del IVA "
            f"({cop(liq.iva_pago)})."
        )
    else:
        lineas.append("ReteIVA: no aplica.")

    if liq.aplica_reteica and liq.regla_ica:
        lineas.append(
            f"ReteICA: {cop(liq.reteica)} = {porcentaje(liq.tarifa_reteica)} "
            f"({liq.regla_ica.municipio}, {liq.regla_ica.actividad})."
        )
    else:
        lineas.append("ReteICA: no aplica.")

    lineas.extend(
        [
            f"Total retenciones: {cop(liq.total_retenciones)}",
            f"Neto a pagar: {cop(liq.neto)}",
            f"Escenario de la tabla: {liq.fila_decision.numero if liq.fila_decision else 'no tabulado'}",
        ]
    )
    return "\n".join(lineas)

## 8. Datos de prueba

Los 12 casos de la sección 10 del diseño funcional. Cada caso usa un
comprador y un proveedor ficticios. El caso 5 sigue el diseño funcional
(comprador SAS normal, sin reteIVA); el README tiene una variante con gran
contribuyente que se prueba aparte en los criterios de aceptación.

In [10]:
@dataclass(frozen=True)
class CasoPrueba:
    numero: int
    nombre: str
    comprador: Comprador
    proveedor: Proveedor
    operacion: Operacion
    esperado: dict[str, Decimal]


def comprador_sas(
    *,
    agente_reteiva: bool = False,
    agente_reteica: bool = False,
    gran_contribuyente: bool = False,
    municipio: str = "Bogotá",
) -> Comprador:
    return Comprador(
        nit="900.123.456-1",
        razon_social="Comprador SAS",
        tipo=TipoPersona.JURIDICA,
        agente_retefuente=True,
        agente_reteiva=agente_reteiva,
        agente_reteica=agente_reteica,
        municipio=municipio,
        responsable_iva=True,
        obligado_contabilidad=True,
        regimen=Regimen.ORDINARIO,
        direccion="Calle 1 # 2-3",
        gran_contribuyente=gran_contribuyente,
    )


def comprador_gran(*, agente_reteica: bool = False) -> Comprador:
    return comprador_sas(agente_reteiva=True, agente_reteica=agente_reteica, gran_contribuyente=True)


def proveedor_declarante(**cambios: Any) -> Proveedor:
    base = dict(
        nit="800.987.654-2",
        razon_social="Proveedor Declarante SAS",
        tipo=TipoPersona.JURIDICA,
        residente=True,
        declarante=True,
        responsable_iva=True,
        simple=False,
        autorretenedor=False,
        autorretencion_especial=False,
        gran_contribuyente=False,
        actividad_ciiu="6201",
        municipio="Bogotá",
        rut_recibido=True,
        soporte_no_declarante=True,
        actividad_gravada_ica=True,
    )
    base.update(cambios)
    return Proveedor(**base)


def proveedor_no_declarante(**cambios: Any) -> Proveedor:
    base = dict(
        nit="1.012.345.678",
        razon_social="Proveedor Persona Natural",
        tipo=TipoPersona.NATURAL,
        residente=True,
        declarante=False,
        responsable_iva=True,
        simple=False,
        autorretenedor=False,
        autorretencion_especial=False,
        gran_contribuyente=False,
        actividad_ciiu="7410",
        municipio="Bogotá",
        rut_recibido=True,
        soporte_no_declarante=True,
        actividad_gravada_ica=True,
    )
    base.update(cambios)
    return Proveedor(**base)


def proveedor_simple(**cambios: Any) -> Proveedor:
    base = dict(
        nit="901.555.000-4",
        razon_social="Proveedor SIMPLE SAS",
        tipo=TipoPersona.JURIDICA,
        residente=True,
        declarante=False,
        responsable_iva=True,
        simple=True,
        autorretenedor=False,
        autorretencion_especial=False,
        gran_contribuyente=False,
        actividad_ciiu="6202",
        municipio="Bogotá",
        rut_recibido=True,
        soporte_no_declarante=True,
        actividad_gravada_ica=True,
    )
    base.update(cambios)
    return Proveedor(**base)


def proveedor_autorretenedor(**cambios: Any) -> Proveedor:
    base = dict(
        nit="900.777.111-5",
        razon_social="Proveedor Autorretenedor SAS",
        tipo=TipoPersona.JURIDICA,
        residente=True,
        declarante=True,
        responsable_iva=True,
        simple=False,
        autorretenedor=True,
        autorretencion_especial=False,
        gran_contribuyente=False,
        actividad_ciiu="6203",
        municipio="Bogotá",
        rut_recibido=True,
        soporte_no_declarante=True,
        actividad_gravada_ica=True,
    )
    base.update(cambios)
    return Proveedor(**base)


def operacion(
    numero: str,
    concepto: Concepto = Concepto.SERVICIO_GENERAL,
    *,
    base: Any | None = None,
    iva: Any | None = None,
    total: Any | None = None,
    tarifa_iva: Any = "0.19",
    total_incluye_iva: bool = False,
    iva_excluido: bool = False,
    descuentos: Any = "0",
    valor_pagado: Any | None = None,
    fecha_emision: date = date(2026, 10, 15),
    fecha_pago: date = date(2026, 10, 15),
    **cambios: Any,
) -> Operacion:
    datos = dict(
        tipo_documento=TipoDocumento.FACTURA,
        numero=numero,
        fecha_emision=fecha_emision,
        fecha_pago=fecha_pago,
        concepto=concepto,
        descripcion=f"Operación de prueba {numero}",
        base=D(base) if base is not None else None,
        tarifa_iva=D(tarifa_iva),
        iva=D(iva) if iva is not None else None,
        total=D(total) if total is not None else None,
        total_incluye_iva=total_incluye_iva,
        descuentos=D(descuentos),
        moneda="COP",
        trm=None,
        forma_pago="Contado",
        valor_pagado=D(valor_pagado) if valor_pagado is not None else None,
        retencion_ya_practicada=False,
        factura_validada=True,
        orden_compra=True,
        acta_recibido=True,
        periodo_cerrado=False,
        iva_excluido=iva_excluido,
        centro_costo="ADMIN",
    )
    datos.update(cambios)
    return Operacion(**datos)


def casos_diseno() -> list[CasoPrueba]:
    return [
        CasoPrueba(
            1,
            "Servicio declarante",
            comprador_sas(),
            proveedor_declarante(),
            operacion("FV-001", base=2_000_000),
            {"iva_pago": D("380000"), "retefuente": D("80000"), "reteiva": CERO, "reteica": CERO, "neto": D("2300000")},
        ),
        CasoPrueba(
            2,
            "Servicio no declarante",
            comprador_sas(),
            proveedor_no_declarante(),
            operacion("FV-002", base=500_000),
            {"iva_pago": D("95000"), "retefuente": D("30000"), "reteiva": CERO, "reteica": CERO, "neto": D("565000")},
        ),
        CasoPrueba(
            3,
            "Bajo mínimo",
            comprador_sas(),
            proveedor_no_declarante(),
            operacion("FV-003", base=90_000),
            {"iva_pago": D("17100"), "retefuente": CERO, "reteiva": CERO, "reteica": CERO, "neto": D("107100")},
        ),
        CasoPrueba(
            4,
            "Exactamente 2 UVT",
            comprador_gran(),
            proveedor_declarante(),
            operacion("FV-004", base=D("104748")),
            {
                "iva_pago": D("19902.12"),
                "retefuente": D("4189.92"),
                "reteiva": D("2985.32"),
                "reteica": CERO,
                "neto": D("117474.88"),
            },
        ),
        CasoPrueba(
            5,
            "Honorarios jurídicos",
            comprador_sas(),
            proveedor_declarante(),
            operacion("FV-005", Concepto.HONORARIOS, base=3_000_000),
            {"iva_pago": D("570000"), "retefuente": D("330000"), "reteiva": CERO, "reteica": CERO, "neto": D("3240000")},
        ),
        CasoPrueba(
            6,
            "Arrendamiento vivienda",
            comprador_sas(),
            proveedor_declarante(),
            operacion("FV-006", Concepto.ARRENDAMIENTO_VIVIENDA, base=1_200_000, tarifa_iva="0", iva_excluido=True),
            {"iva_pago": CERO, "retefuente": D("42000"), "reteiva": CERO, "reteica": CERO, "neto": D("1158000")},
        ),
        CasoPrueba(
            7,
            "Compra declarante",
            comprador_sas(),
            proveedor_declarante(),
            operacion("FV-007", Concepto.COMPRA_BIENES, base=4_000_000),
            {"iva_pago": D("760000"), "retefuente": D("100000"), "reteiva": CERO, "reteica": CERO, "neto": D("4660000")},
        ),
        CasoPrueba(
            8,
            "SIMPLE",
            comprador_gran(),
            proveedor_simple(),
            operacion("FV-008", base=1_000_000),
            {"iva_pago": D("190000"), "retefuente": CERO, "reteiva": D("28500"), "reteica": CERO, "neto": D("1161500")},
        ),
        CasoPrueba(
            9,
            "Autorretenedor",
            comprador_gran(),
            proveedor_autorretenedor(),
            operacion("FV-009", base=1_000_000),
            {"iva_pago": D("190000"), "retefuente": CERO, "reteiva": D("28500"), "reteica": CERO, "neto": D("1161500")},
        ),
        CasoPrueba(
            10,
            "Total con IVA",
            comprador_sas(),
            proveedor_declarante(),
            operacion("FV-010", total=1_190_000, total_incluye_iva=True),
            {"base_pago": D("1000000"), "iva_pago": D("190000"), "retefuente": D("40000"), "reteiva": CERO, "reteica": CERO, "neto": D("1150000")},
        ),
        CasoPrueba(
            11,
            "Pago parcial",
            comprador_gran(),
            proveedor_declarante(),
            operacion("FV-011", base=1_000_000, valor_pagado=595_000),
            {
                "base_pago": D("500000"),
                "iva_pago": D("95000"),
                "total_pago": D("595000"),
                "retefuente": D("20000"),
                "reteiva": D("14250"),
                "reteica": CERO,
                "neto": D("560750"),
            },
        ),
        CasoPrueba(
            12,
            "ReteICA Bogotá",
            comprador_sas(agente_reteica=True),
            proveedor_declarante(),
            operacion("FV-012", base=2_000_000),
            {"iva_pago": D("380000"), "retefuente": D("80000"), "reteiva": CERO, "reteica": D("19320"), "neto": D("2280680")},
        ),
    ]

## 9. Ejecución de los 12 casos

La columna **Escenario** indica la fila pedagógica de la tabla de decisión.
No significa que la retención se haya practicado: el mínimo puede dejarla en
cero. Los valores de retención están en las columnas de cada concepto.

In [11]:
def ejecutar_casos(casos: Sequence[CasoPrueba]) -> list[tuple[CasoPrueba, Liquidacion]]:
    resultados: list[tuple[CasoPrueba, Liquidacion]] = []
    for caso in casos:
        liquidador = LiquidadorRetenciones(usuario="prueba.automatica")
        liquidacion = liquidador.calcular(caso.comprador, caso.proveedor, caso.operacion)
        resultados.append((caso, liquidacion))
    return resultados


def comparar_caso(caso: CasoPrueba, liquidacion: Liquidacion) -> list[str]:
    errores: list[str] = []
    for campo, esperado in caso.esperado.items():
        obtenido = getattr(liquidacion, campo)
        if dinero(obtenido) != dinero(esperado):
            errores.append(f"{campo}: esperado {cop(esperado)}, obtenido {cop(obtenido)}")
    return errores


RESULTADOS = ejecutar_casos(casos_diseno())
mostrar(tabla_resultados(RESULTADOS))

ERRORES_CASOS: list[str] = []
for caso, liquidacion in RESULTADOS:
    errores = comparar_caso(caso, liquidacion)
    if errores:
        ERRORES_CASOS.append(f"Caso {caso.numero}: " + "; ".join(errores))

if ERRORES_CASOS:
    print("❌ Casos con diferencias:")
    for error in ERRORES_CASOS:
        print(" -", error)
else:
    print("✅ Los 12 casos del diseño funcional coinciden con los valores esperados.")

,#,Caso,Base pago,IVA,Total pago,Retefuente,ReteIVA,ReteICA,Neto,Escenario
0,1,Servicio declarante,"$2.000.000,00","$380.000,00","$2.380.000,00","$80.000,00","$0,00","$0,00","$2.300.000,00",1
1,2,Servicio no declarante,"$500.000,00","$95.000,00","$595.000,00","$30.000,00","$0,00","$0,00","$565.000,00",2
2,3,Bajo mínimo,"$90.000,00","$17.100,00","$107.100,00","$0,00","$0,00","$0,00","$107.100,00",2
3,4,Exactamente 2 UVT,"$104.748,00","$19.902,12","$124.650,12","$4.189,92","$2.985,32","$0,00","$117.474,88",8
4,5,Honorarios jurídicos,"$3.000.000,00","$570.000,00","$3.570.000,00","$330.000,00","$0,00","$0,00","$3.240.000,00",3
5,6,Arrendamiento vivienda,"$1.200.000,00","$0,00","$1.200.000,00","$42.000,00","$0,00","$0,00","$1.158.000,00",5
6,7,Compra declarante,"$4.000.000,00","$760.000,00","$4.760.000,00","$100.000,00","$0,00","$0,00","$4.660.000,00",6
7,8,SIMPLE,"$1.000.000,00","$190.000,00","$1.190.000,00","$0,00","$28.500,00","$0,00","$1.161.500,00",12
8,9,Autorretenedor,"$1.000.000,00","$190.000,00","$1.190.000,00","$0,00","$28.500,00","$0,00","$1.161.500,00",13
9,10,Total con IVA,"$1.000.000,00","$190.000,00","$1.190.000,00","$40.000,00","$0,00","$0,00","$1.150.000,00",1


✅ Los 12 casos del diseño funcional coinciden con los valores esperados.


### 9.1 Explicación del caso 1 (sección 9 del diseño)

In [12]:
CASO_1, LIQ_1 = RESULTADOS[0]
print(explicar(LIQ_1))

Concepto: Servicio general
Base factura: $2.000.000,00
IVA: $380.000,00 (19%)
Total factura: $2.380.000,00
Mínimo retefuente: 2 UVT = $104.748,00 → cumple
Tarifa retefuente: 4% (declarante)
Retefuente: $80.000,00
Fundamento: Art. 392 ET; Decreto 572/2025; efectos desde 1/07/2026; UVT 2026.
ReteIVA: no aplica.
ReteICA: no aplica.
Total retenciones: $80.000,00
Neto a pagar: $2.300.000,00
Escenario de la tabla: 1


### 9.2 Asientos del caso 1

In [13]:
for asiento in LIQ_1.asientos:
    print(f"\n{asiento.tipo} — {asiento.documento} — {asiento.fecha}")
    mostrar(tabla_asiento(asiento))
    print(f"Débitos: {cop(asiento.total_debito)} | Créditos: {cop(asiento.total_credito)} | Diferencia: {cop(asiento.diferencia)}")


Causación — FV-001 — 2026-10-15


,Cuenta,Nombre,Débito,Crédito,Tercero,Centro de costo,Documento
0,5135,Servicios,"$2.000.000,00",,Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001
1,240805,IVA descontable,"$380.000,00",,Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001
2,220505,Proveedores nacionales,,"$2.380.000,00",Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001


Débitos: $2.380.000,00 | Créditos: $2.380.000,00 | Diferencia: $0,00

Pago — FV-001 — 2026-10-15


,Cuenta,Nombre,Débito,Crédito,Tercero,Centro de costo,Documento
0,220505,Proveedores nacionales,"$2.380.000,00",,Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001
1,111005,Bancos,,"$2.300.000,00",Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001
2,236525,Retención en la fuente — servicios,,"$80.000,00",Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001


Débitos: $2.380.000,00 | Créditos: $2.380.000,00 | Diferencia: $0,00

Pago de retenciones — FV-001 — 2026-10-15


,Cuenta,Nombre,Débito,Crédito,Tercero,Centro de costo,Documento
0,236525,Retención en la fuente — servicios,"$80.000,00",,Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001
1,111005,Bancos,,"$80.000,00",Proveedor Declarante SAS / 800.987.654-2,ADMIN,FV-001


Débitos: $80.000,00 | Créditos: $80.000,00 | Diferencia: $0,00


### 9.3 Certificados y checklist del caso 1

In [14]:
for certificado in LIQ_1.certificados:
    print(f"{certificado.tipo}: {certificado.numero} | base {cop(certificado.base)} | "
          f"tarifa {porcentaje(certificado.tarifa)} | valor {cop(certificado.valor)}")

print("\nChecklist:")
for item in LIQ_1.checklist:
    estado = "✔" if item.recibido else "✘"
    print(f" {estado} {item.documento}" + (f" — {item.observacion}" if item.observacion else ""))

Renta: REN-FV-001-2026 | base $2.000.000,00 | tarifa 4% | valor $80.000,00

Checklist:
 ✔ RUT del proveedor
 ✔ Factura electrónica validada
 ✔ Orden de compra
 ✔ Acta de recibido
 ✔ Cálculo de retención — Generado por el motor.
 ✔ Asiento contable
 ✔ Certificado de renta — Art. 381 ET.
 ✔ Borrador Formulario 350 — No se presenta automáticamente.


### 9.4 Auditoría del caso 1

In [15]:
auditoria = LIQ_1.auditoria
print(f"Usuario: {auditoria.usuario}")
print(f"Fecha y hora: {auditoria.fecha_hora:%Y-%m-%d %H:%M:%S}")
print(f"Versión de tablas: {auditoria.version_tabla}")
print(f"UVT: {cop(auditoria.uvt)} ({auditoria.uvt_anio})")
print(f"Regla: {auditoria.regla_id} | Fundamento: {auditoria.fundamento}")
print(f"Hash de entrada: {auditoria.hash_entrada[:24]}...")
print(f"Documentos: {', '.join(auditoria.documentos) or 'ninguno'}")

Usuario: prueba.automatica
Fecha y hora: 2026-10-28 16:11:31
Versión de tablas: 2026.1
UVT: $52.374,00 (2026)
Regla: RF-SERV-005 | Fundamento: Art. 392 ET; Decreto 572/2025; efectos desde 1/07/2026; UVT 2026.
Hash de entrada: 472f5c6bf2688cdbee4ca8d4...
Documentos: REN-FV-001-2026, RUT del proveedor, Factura electrónica validada, Orden de compra, Acta de recibido, Cálculo de retención, Asiento contable, Certificado de renta, Borrador Formulario 350


### 9.5 Borradores de declaración (sin presentarlos)

El motor no firma ni presenta declaraciones: solo deja el resumen de bases,
tarifas y valores para que el contador arme el Formulario 350 o la
declaración municipal.

In [16]:
CASO_12, LIQ_12 = RESULTADOS[11]
print("Caso 12 — servicio con retefuente y reteICA:")
mostrar(borradores_declaracion(LIQ_12))

CASO_4, LIQ_4 = RESULTADOS[3]
print("\nCaso 4 — servicio con retefuente y reteIVA:")
mostrar(borradores_declaracion(LIQ_4))

Caso 12 — servicio con retefuente y reteICA:


,Declaración,Concepto,Base,Tarifa,Valor
0,Formulario 350,Retefuente,"$2.000.000,00",4%,"$80.000,00"
1,Municipal — Bogotá,ReteICA,"$2.000.000,00","0,966%","$19.320,00"



Caso 4 — servicio con retefuente y reteIVA:


,Declaración,Concepto,Base,Tarifa,Valor
0,Formulario 350,Retefuente,"$104.748,00",4%,"$4.189,92"
1,Formulario 350,ReteIVA,"$19.902,12",15%,"$2.985,32"


## 10. Pruebas de aceptación

Se verifican los 10 criterios de la sección 11 del diseño, además de los
asientos, certificados y bloqueos. Las pruebas usan ``assert`` para que el
notebook falle de forma visible si una regla se rompe.

In [17]:
def assert_dinero(obtenido: Any, esperado: Any, mensaje: str) -> None:
    assert dinero(obtenido) == dinero(esperado), f"{mensaje}: esperado {cop(esperado)}, obtenido {cop(obtenido)}"


def liq_caso(numero: int) -> Liquidacion:
    for caso, liquidacion in RESULTADOS:
        if caso.numero == numero:
            return liquidacion
    raise KeyError(numero)


# 1. Servicio de $2.000.000 + IVA 19% de proveedor declarante, comprador SAS normal.
L = liq_caso(1)
assert_dinero(L.iva_pago, "380000", "IVA caso 1")
assert_dinero(L.retefuente, "80000", "RF caso 1")
assert_dinero(L.neto, "2300000", "Neto caso 1")

# 2. El mismo caso con comprador gran contribuyente: reteIVA 15% del IVA.
CASO_ACEPTACION_2 = CasoPrueba(
    101,
    "Servicio con comprador gran contribuyente",
    comprador_gran(),
    proveedor_declarante(),
    operacion("FV-101", base=2_000_000),
    {"reteiva": D("57000"), "neto": D("2243000")},
)
LIQ_ACEPTACION_2 = LiquidadorRetenciones(usuario="prueba.aceptacion").calcular(
    CASO_ACEPTACION_2.comprador, CASO_ACEPTACION_2.proveedor, CASO_ACEPTACION_2.operacion
)
assert_dinero(LIQ_ACEPTACION_2.reteiva, "57000", "ReteIVA aceptación 2")
assert_dinero(LIQ_ACEPTACION_2.neto, "2243000", "Neto aceptación 2")

# 3. Servicio de $90.000: no hay retefuente ni reteIVA, aunque el comprador sea agente.
CASO_ACEPTACION_3 = CasoPrueba(
    102,
    "Servicio bajo mínimo con comprador gran contribuyente",
    comprador_gran(),
    proveedor_declarante(),
    operacion("FV-102", base=90_000),
    {"retefuente": CERO, "reteiva": CERO, "neto": D("107100")},
)
LIQ_ACEPTACION_3 = LiquidadorRetenciones(usuario="prueba.aceptacion").calcular(
    CASO_ACEPTACION_3.comprador, CASO_ACEPTACION_3.proveedor, CASO_ACEPTACION_3.operacion
)
assert_dinero(LIQ_ACEPTACION_3.retefuente, "0", "RF aceptación 3")
assert_dinero(LIQ_ACEPTACION_3.reteiva, "0", "ReteIVA aceptación 3")

# 4. Proveedor SIMPLE: no hay retefuente ni reteICA; reteIVA si es responsable de IVA.
CASO_ACEPTACION_4 = CasoPrueba(
    103,
    "SIMPLE con comprador agente de reteICA",
    comprador_gran(agente_reteica=True),
    proveedor_simple(),
    operacion("FV-103", base=1_000_000),
    {"retefuente": CERO, "reteica": CERO, "reteiva": D("28500")},
)
LIQ_ACEPTACION_4 = LiquidadorRetenciones(usuario="prueba.aceptacion").calcular(
    CASO_ACEPTACION_4.comprador, CASO_ACEPTACION_4.proveedor, CASO_ACEPTACION_4.operacion
)
assert_dinero(LIQ_ACEPTACION_4.retefuente, "0", "RF aceptación 4")
assert_dinero(LIQ_ACEPTACION_4.reteica, "0", "ReteICA aceptación 4")
assert_dinero(LIQ_ACEPTACION_4.reteiva, "28500", "ReteIVA aceptación 4")

# 5. Proveedor autorretenedor: no hay retefuente.
CASO_ACEPTACION_5 = CasoPrueba(
    104,
    "Autorretenedor",
    comprador_gran(),
    proveedor_autorretenedor(),
    operacion("FV-104", base=1_000_000),
    {"retefuente": CERO, "reteiva": D("28500")},
)
LIQ_ACEPTACION_5 = LiquidadorRetenciones(usuario="prueba.aceptacion").calcular(
    CASO_ACEPTACION_5.comprador, CASO_ACEPTACION_5.proveedor, CASO_ACEPTACION_5.operacion
)
assert_dinero(LIQ_ACEPTACION_5.retefuente, "0", "RF aceptación 5")

# 6. Todos los asientos cuadran.
for caso, liquidacion in RESULTADOS:
    assert liquidacion.asientos_cuadran, f"El asiento del caso {caso.numero} no cuadra."
    for asiento in liquidacion.asientos:
        assert abs(asiento.diferencia) < D("0.005"), f"Asiento descuadrado: {caso.numero} / {asiento.tipo}"

# 7. El certificado de renta cumple los siete requisitos del Art. 381.
cert_renta = next(c for c in liq_caso(1).certificados if c.tipo == "Renta")
requisitos = cert_renta.requisitos_art_381()
assert len(requisitos) == 7
assert all(valor is not None and str(valor).strip() != "" for valor in requisitos.values())

# 8. El certificado de IVA discrimina base sin IVA, IVA, tarifa y valor.
cert_iva = next(c for c in liq_caso(4).certificados if c.tipo == "ReteIVA")
assert cert_iva.base_sin_iva == D("104748.00")
assert cert_iva.iva == D("19902.12")
assert cert_iva.tarifa == D("0.15")
assert cert_iva.valor == D("2985.32")

# 9. Se guarda la versión de la tabla y la UVT usada.
for _, liquidacion in RESULTADOS:
    assert liquidacion.auditoria is not None
    assert liquidacion.auditoria.version_tabla == VERSION_TABLAS
    assert liquidacion.auditoria.uvt == UVT_2026
    assert liquidacion.auditoria.uvt_anio == 2026

# 10. Se bloquea el cálculo si falta una regla o un dato crítico.
def debe_bloquear(comprador: Comprador, proveedor: Proveedor, op: Operacion) -> str:
    try:
        LiquidadorRetenciones(usuario="prueba.bloqueo").calcular(comprador, proveedor, op)
    except CalculoBloqueado as error:
        return error.mensaje
    raise AssertionError("Se esperaba un bloqueo y el cálculo se ejecutó.")


mensaje_bloqueo = debe_bloquear(
    comprador_sas(),
    proveedor_declarante(),
    operacion("FV-105", base=2_000_000, periodo_cerrado=True),
)
assert "cerrado" in mensaje_bloqueo.lower()

mensaje_bloqueo = debe_bloquear(
    comprador_sas(),
    proveedor_declarante(nit=""),
    operacion("FV-106", base=2_000_000),
)
assert "proveedor" in mensaje_bloqueo.lower()

print("✅ Criterios de aceptación 1 a 10 verificados.")
print(f"   - Caso 2 de aceptación: reteIVA {cop(LIQ_ACEPTACION_2.reteiva)}, neto {cop(LIQ_ACEPTACION_2.neto)}")
print(f"   - Caso 3 de aceptación: RF {cop(LIQ_ACEPTACION_3.retefuente)}, reteIVA {cop(LIQ_ACEPTACION_3.reteiva)}")
print(f"   - Caso 4 de aceptación: RF {cop(LIQ_ACEPTACION_4.retefuente)}, reteICA {cop(LIQ_ACEPTACION_4.reteica)}")
print(f"   - Caso 5 de aceptación: RF {cop(LIQ_ACEPTACION_5.retefuente)}")

✅ Criterios de aceptación 1 a 10 verificados.
   - Caso 2 de aceptación: reteIVA $57.000,00, neto $2.243.000,00
   - Caso 3 de aceptación: RF $0,00, reteIVA $0,00
   - Caso 4 de aceptación: RF $0,00, reteICA $0,00
   - Caso 5 de aceptación: RF $0,00


## 11. Casos de bloqueo y revisión humana

Estos casos no se calculan: se derivan a revisión humana o se bloquean. Es
preferible no calcular que calcular mal.

In [18]:
casos_bloqueo = [
    (
        "Proveedor no residente",
        comprador_sas(),
        proveedor_declarante(residente=False),
        operacion("BLQ-001", base=2_000_000),
    ),
    (
        "Factura no validada",
        comprador_sas(),
        proveedor_declarante(),
        operacion("BLQ-002", base=2_000_000, factura_validada=False),
    ),
    (
        "Periodo cerrado",
        comprador_sas(),
        proveedor_declarante(),
        operacion("BLQ-003", base=2_000_000, periodo_cerrado=True),
    ),
    (
        "Sin soporte de no declarante",
        comprador_sas(),
        proveedor_no_declarante(soporte_no_declarante=False),
        operacion("BLQ-004", base=2_000_000),
    ),
    (
        "Sin tarifa ICA para el municipio",
        comprador_sas(agente_reteica=True, municipio="Medellín"),
        proveedor_declarante(municipio="Medellín"),
        operacion("BLQ-005", base=2_000_000),
    ),
    (
        "Sin UVT para el año",
        comprador_sas(),
        proveedor_declarante(),
        operacion("BLQ-006", base=2_000_000, fecha_emision=date(2027, 1, 15), fecha_pago=date(2027, 1, 15)),
    ),
]

for nombre, comprador, proveedor, op in casos_bloqueo:
    try:
        LiquidadorRetenciones(usuario="prueba.bloqueo").calcular(comprador, proveedor, op)
    except CalculoBloqueado as error:
        print(f"🔒 {nombre}: {error.mensaje}")
    else:
        print(f"⚠️ {nombre}: no se bloqueó (revisar).")

🔒 Proveedor no residente: No se puede liquidar: Proveedor no residente: el caso se deriva al módulo de pagos al exterior (fuera de la v1).
🔒 Factura no validada: No se puede liquidar: La factura no está validada por la DIAN: se bloquea el IVA y el costo.
🔒 Periodo cerrado: No se puede liquidar: El periodo está cerrado; se requiere un ajuste autorizado.
🔒 Sin soporte de no declarante: No se puede liquidar: No hay certificación de no declarante. Use la tarifa conservadora o bloquee; la v1 bloquea.
🔒 Sin tarifa ICA para el municipio: No hay tarifa de reteICA para 'Medellín', actividad 'Servicios', el 2026-10-15. No se inventa la tarifa.
🔒 Sin UVT para el año: No hay UVT cargada para el año 2027.


## 12. Duplicados

El repositorio de facturas evita dos cálculos para el mismo proveedor y
número. La autorización explícita permite recalcular, pero deja constancia.

In [19]:
repositorio = RepositorioFacturas()
comprador_dup = comprador_sas()
proveedor_dup = proveedor_declarante()
operacion_dup = operacion("FV-DUP-001", base=2_000_000)

primera = LiquidadorRetenciones(usuario="usuario.1", repositorio=repositorio).calcular(
    comprador_dup, proveedor_dup, operacion_dup
)
print(f"Primera liquidación: {primera.operacion.numero} por {cop(primera.neto)}")

try:
    LiquidadorRetenciones(usuario="usuario.2", repositorio=repositorio).calcular(
        comprador_dup, proveedor_dup, operacion_dup
    )
except CalculoBloqueado as error:
    print(f"🔒 Segundo cálculo bloqueado: {error.mensaje}")

segunda = LiquidadorRetenciones(usuario="usuario.3", repositorio=repositorio).calcular(
    comprador_dup, proveedor_dup, operacion_dup, autorizar_duplicado=True
)
print(f"Recálculo autorizado: {segunda.operacion.numero} por {cop(segunda.neto)}")

Primera liquidación: FV-DUP-001 por $2.300.000,00
🔒 Segundo cálculo bloqueado: Posible factura duplicada: ya existe FV-DUP-001 del proveedor 800.987.654-2.
Recálculo autorizado: FV-DUP-001 por $2.300.000,00


## 13. Variantes que conviene tener a mano

Estas variantes muestran que la tabla de decisión no es una fórmula única:
la misma base cambia según el comprador y el proveedor.

In [20]:
variantes = [
    (
        "SAS normal, declarante",
        comprador_sas(),
        proveedor_declarante(),
        operacion("VAR-001", base=2_000_000),
    ),
    (
        "Gran contribuyente, declarante",
        comprador_gran(),
        proveedor_declarante(),
        operacion("VAR-002", base=2_000_000),
    ),
    (
        "SAS normal, no declarante",
        comprador_sas(),
        proveedor_no_declarante(),
        operacion("VAR-003", base=2_000_000),
    ),
    (
        "Gran contribuyente, SIMPLE",
        comprador_gran(),
        proveedor_simple(),
        operacion("VAR-004", base=2_000_000),
    ),
    (
        "Gran contribuyente, autorretenedor",
        comprador_gran(),
        proveedor_autorretenedor(),
        operacion("VAR-005", base=2_000_000),
    ),
]

filas_variantes = []
for nombre, comprador, proveedor, op in variantes:
    liquidacion = LiquidadorRetenciones(usuario="prueba.variantes").calcular(comprador, proveedor, op)
    filas_variantes.append(
        {
            "Variante": nombre,
            "Escenario": liquidacion.fila_decision.numero if liquidacion.fila_decision else "—",
            "RF": cop(liquidacion.retefuente),
            "ReteIVA": cop(liquidacion.reteiva),
            "ReteICA": cop(liquidacion.reteica),
            "Neto": cop(liquidacion.neto),
        }
    )

if pd is not None:
    mostrar(pd.DataFrame(filas_variantes))
else:
    for fila in filas_variantes:
        print(fila)

,Variante,Escenario,RF,ReteIVA,ReteICA,Neto
0,"SAS normal, declarante",1,"$80.000,00","$0,00","$0,00","$2.300.000,00"
1,"Gran contribuyente, declarante",8,"$80.000,00","$57.000,00","$0,00","$2.243.000,00"
2,"SAS normal, no declarante",2,"$120.000,00","$0,00","$0,00","$2.260.000,00"
3,"Gran contribuyente, SIMPLE",12,"$0,00","$57.000,00","$0,00","$2.323.000,00"
4,"Gran contribuyente, autorretenedor",13,"$0,00","$57.000,00","$0,00","$2.323.000,00"


## 14. Nota crédito simple

El diseño incluye notas crédito simples. Para no mezclar una nota con una
factura, se modela como un ajuste sobre una liquidación no pagada: reduce
base, IVA y retenciones de forma proporcional. Si la factura ya se pagó, el
ajuste debe ir al siguiente periodo y requiere revisión humana.

In [21]:
@dataclass(frozen=True)
class AjusteNotaCredito:
    referencia: str
    base: Decimal
    iva: Decimal
    total: Decimal
    retefuente_reversada: Decimal
    reteiva_reversada: Decimal
    reteica_reversada: Decimal
    neto_ajustado: Decimal
    asiento: Asiento


def calcular_nota_credito(
    liquidacion: Liquidacion,
    numero: str,
    base_nc: Any,
    iva_nc: Any,
    fecha: date,
) -> AjusteNotaCredito:
    if liquidacion.factor_pago != 1:
        raise RevisionHumana("La nota crédito sobre un pago parcial no está cubierta por la v1.")
    base_nc = dinero(base_nc)
    iva_nc = dinero(iva_nc)
    total_nc = dinero(base_nc + iva_nc)
    if base_nc <= 0 or iva_nc < 0:
        raise CalculoBloqueado("La nota crédito debe tener base positiva e IVA no negativo.")
    if base_nc > liquidacion.datos.base or iva_nc > liquidacion.datos.iva:
        raise CalculoBloqueado("La nota crédito no puede superar la base o el IVA de la factura original.")

    factor_nc = base_nc / liquidacion.datos.base
    rf_rev = dinero(liquidacion.retefuente * factor_nc)
    riva_rev = dinero(liquidacion.reteiva * factor_nc)
    rica_rev = dinero(liquidacion.reteica * factor_nc)
    neto_ajustado = dinero(liquidacion.neto - total_nc + rf_rev + riva_rev + rica_rev)

    tercero = f"{liquidacion.proveedor.razon_social} / {liquidacion.proveedor.nit}"
    cuenta, nombre_cuenta = cuenta_gasto(liquidacion.operacion.concepto)
    lineas = [
        LineaAsiento(CUENTAS["proveedores"][0], CUENTAS["proveedores"][1], total_nc, CERO, tercero, liquidacion.operacion.centro_costo, "Nota crédito", numero),
        LineaAsiento(cuenta, nombre_cuenta, CERO, base_nc, tercero, liquidacion.operacion.centro_costo, "Reverso de base", numero),
    ]
    if iva_nc > 0:
        lineas.append(
            LineaAsiento(CUENTAS["iva_descontable"][0], CUENTAS["iva_descontable"][1], CERO, iva_nc, tercero, liquidacion.operacion.centro_costo, "Reverso de IVA", numero)
        )
    asiento = Asiento("Nota crédito", fecha, numero, "Ajuste por nota crédito", lineas)
    if not asiento.cuadra:
        raise CalculoBloqueado("El asiento de la nota crédito no cuadra.")

    return AjusteNotaCredito(
        referencia=liquidacion.operacion.numero,
        base=base_nc,
        iva=iva_nc,
        total=total_nc,
        retefuente_reversada=rf_rev,
        reteiva_reversada=riva_rev,
        reteica_reversada=rica_rev,
        neto_ajustado=neto_ajustado,
        asiento=asiento,
    )


nota = calcular_nota_credito(
    LIQ_1,
    "NC-001",
    base_nc=1_000_000,
    iva_nc=190_000,
    fecha=date(2026, 10, 20),
)
print(f"Nota crédito {nota.referencia}: base {cop(nota.base)}, IVA {cop(nota.iva)}, total {cop(nota.total)}")
print(f"RF reversada: {cop(nota.retefuente_reversada)} | ReteIVA reversada: {cop(nota.reteiva_reversada)}")
print(f"Neto ajustado: {cop(nota.neto_ajustado)}")
mostrar(tabla_asiento(nota.asiento))

Nota crédito FV-001: base $1.000.000,00, IVA $190.000,00, total $1.190.000,00
RF reversada: $40.000,00 | ReteIVA reversada: $0,00
Neto ajustado: $1.150.000,00


,Cuenta,Nombre,Débito,Crédito,Tercero,Centro de costo,Documento
0,220505,Proveedores nacionales,"$1.190.000,00",,Proveedor Declarante SAS / 800.987.654-2,ADMIN,NC-001
1,5135,Servicios,,"$1.000.000,00",Proveedor Declarante SAS / 800.987.654-2,ADMIN,NC-001
2,240805,IVA descontable,,"$190.000,00",Proveedor Declarante SAS / 800.987.654-2,ADMIN,NC-001


## 15. Limitaciones y siguientes pasos

Este notebook implementa la v1 del diseño funcional. No cubre:

- Pagos al exterior ni convenios de doble tributación.
- Régimen SIMPLE a fondo; solo la bandera de exclusión.
- Autorretención especial.
- Hidrocarburos, construcción, entidades públicas y salud especial.
- Servicios temporales, aseo y vigilancia con AIU.
- Operaciones entre grandes contribuyentes con reglas especiales.
- Prorrata de IVA.
- Moneda extranjera con diferencia en cambio.
- Declaraciones automáticas ante la DIAN.
- Planeación tributaria.

Además, esta implementación didáctica:

- Solo incluye la tarifa pedagógica de ICA de Bogotá para servicios; otras
  actividades o municipios bloquean por falta de regla.
- Cubre los conceptos del diseño y algunos de la tabla de referencia, no las
  77 filas de la tabla oficial.
- No firma ni presenta declaraciones: genera borradores y deja la decisión
  final al contador.

**Regla de oro:** si el caso no está cubierto, el motor bloquea y pide
revisión humana. La tabla, la UVT y la fecha de vigencia son parte de la
regla; no se deben separar del cálculo.

Para producción faltaría:

1. Cargar las tablas oficiales completas y sus vigencias.
2. Validar el RUT y la calidad de agente contra fuentes oficiales.
3. Persistir facturas, liquidaciones y auditoría en una base de datos.
4. Agregar autenticación, roles y bloqueo de periodos.
5. Probar con 20 facturas reales y firma del contador.
6. Ejecutar la marcha blanca en paralelo con el proceso manual.